# GICN (Faithful)

In [ ]:

# =======================================
# GICN / "Improved GCN" (Zhu et al., 2022) — reproduction-oriented implementation (STRICT)
#
# Goals:
#   - Subject-wise 10-fold Stratified CV (no subject leakage)
#   - Segment-level samples (4s window, 0.5s shift)
#   - Adjacency = |Pearson corr|, then D^{-1/2} A D^{-1/2}
#   - Node features = [activity, mobility, complexity, PSD]
#   - Model = learnable W_alpha in input layer + 2 GCN + Dense(6) + Dropout(0.2)
#
# This notebook intentionally keeps "degrees of freedom" configurable because the paper
# omits some details (e.g., AR order for PSD, exact preprocessing parameters).
# =======================================

import os, re, glob, zipfile, random, math, copy
import numpy as np

from scipy.io import loadmat
from scipy.signal import welch, firwin, filtfilt

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, confusion_matrix, roc_auc_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs): return x

# -----------------------
# 0) Paths / config
# -----------------------
ZIP_PATH  = "/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015.zip"
DATA_ROOT = "/content/modma_lanzhou"

# Strongly recommended: provide explicit labels to avoid filename heuristics.
# CSV format: subject_id,label   (label: 1=MDD, 0=HC)
LABEL_CSV = None  # e.g., "/content/modma_lanzhou/labels.csv"

# If multiple .mat files exist per subject_id:
GROUP_BY_SUBJECT_ID = True
CONCAT_FILES_PER_SUBJECT = True  # if False: use only the first file per subject

# Preprocessing (paper describes MATLAB/EEGLAB steps; here we implement at least bandpass)
APPLY_BANDPASS = True
BANDPASS_LO, BANDPASS_HI = 1.0, 40.0
FIR_TAPS = 401  # odd number recommended
FS = 250

# Segmentation.
# NOTE: The paper says "first 3 min", but also reports 393 segments with (4s, 0.5s shift),
# which corresponds to 200 seconds. We keep MAX_SEC=200 to match the reported sample count.
WIN_SEC  = 4.0
STEP_SEC = 0.5
MAX_SEC  = 200.0   # set 180.0 if you want the literal "3 min" interpretation

# Graph construction
ADJ_PER_SEGMENT = True  # if False: compute one adjacency per subject (sensitivity analysis)

# PSD method (paper uses AR-based PSD but does not specify AR order)
PSD_METHOD = "ar"   # "ar" or "welch"
AR_ORDER   = 16     # IMPORTANT degree-of-freedom (not specified in paper)
PSD_FMIN, PSD_FMAX = 1.0, 40.0
PSD_NFREQ  = 256

# Training
SEED = 42
N_SPLITS = 10
VAL_RATIO = 0.0     # subject-wise validation inside train fold (0 disables; paper doesn't mention a val split)     # subject-wise validation inside train fold (0 disables)
NUM_EPOCHS = 60
BATCH_SIZE = 64
BASE_LR = 0.005
LR_STEP = 30
LR_GAMMA = 0.1
DROPOUT = 0.2
DENSE_DIM = 6

# Optional (OFF by default): normalize node features using training subjects only per fold
NORMALIZE_NODE_FEATURES = False

# Evaluation
EVAL_SUBJECT_MAJORITY = False   # In the paper, evaluation appears segment-level; keep False by default.

# -----------------------
# 1) Helpers
# -----------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def ensure_unzipped(zip_path: str, dst_root: str):
    os.makedirs(dst_root, exist_ok=True)
    mat_glob = glob.glob(os.path.join(dst_root, "**", "*.mat"), recursive=True)
    if mat_glob:
        print(f"Found {len(mat_glob)} .mat files under {dst_root} (skip unzip).")
        return
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Zip not found: {zip_path}\nFix ZIP_PATH.")
    print("Extracting zip... (first run only)")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dst_root)
    mat_glob = glob.glob(os.path.join(dst_root, "**", "*.mat"), recursive=True)
    print(f"Done. Found {len(mat_glob)} .mat files under {dst_root}.")

def load_label_map(csv_path: str):
    m = {}
    with open(csv_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith("subject"):
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 2:
                continue
            sid, lab = parts[0], int(parts[1])
            m[sid] = lab
    return m

def parse_subject_id(path: str):
    fname = os.path.basename(path)
    # Try multiple patterns: MODMA sometimes uses MDD/HC; some Lanzhou files use 02xxxx...
    for pat in [r"(MDD[_-]?\d+)", r"(HC[_-]?\d+)", r"(02\d+)"]:
        m = re.search(pat, fname, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    # fallback: use parent folder name
    return os.path.basename(os.path.dirname(path))

def parse_label(path: str, subject_id: str, label_map=None):
    if label_map is not None and subject_id in label_map:
        return int(label_map[subject_id])
    # Heuristic fallback (NOT recommended)
    low = path.lower()
    if "mdd" in low or "depress" in low:
        return 1
    if "hc" in low or "control" in low or "healthy" in low:
        return 0
    if subject_id.startswith("0201"):
        return 1
    return 0

def load_eeg_mat(path: str):
    """
    Robust loader: finds a 2D array whose one dimension is 128 or 129.
    Returns shape (128, T) float32.
    """
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    candidates = []
    for k, v in mat.items():
        if k.startswith("__"):
            continue
        arr = np.asarray(v)
        if arr.ndim != 2:
            continue
        # prefer arrays that look like EEG
        if arr.shape[0] in (128, 129) or arr.shape[1] in (128, 129):
            candidates.append(arr)
    if not candidates:
        # fallback: first non-private key
        keys = [k for k in mat.keys() if not k.startswith("__")]
        if not keys:
            raise ValueError(f"No data keys in {path}")
        candidates = [np.asarray(mat[keys[0]])]
    # pick the best candidate: closest to 128 channels
    def score(a):
        s0 = min(abs(a.shape[0]-128), abs(a.shape[0]-129))
        s1 = min(abs(a.shape[1]-128), abs(a.shape[1]-129))
        return min(s0, s1)
    x = min(candidates, key=score).astype(np.float32)

    # ensure shape = (channels, time)
    if x.shape[0] in (128, 129):
        pass
    elif x.shape[1] in (128, 129):
        x = x.T
    else:
        # last resort: assume already (C,T)
        pass

    if x.shape[0] == 129:
        x = x[:128, :]
    if x.shape[0] != 128:
        raise ValueError(f"Unexpected EEG channel count {x.shape[0]} in {path} (expected 128/129)")
    return x  # (128, T)

def bandpass_fir(x, fs=FS, lo=BANDPASS_LO, hi=BANDPASS_HI, taps=FIR_TAPS):
    nyq = fs / 2.0
    b = firwin(taps, [lo/nyq, hi/nyq], pass_zero=False)
    return filtfilt(b, [1.0], x, axis=1).astype(np.float32)

# -----------------------
# 2) Channel selection (paper removes 23 electrodes -> keep 105)
# -----------------------
DROP_E = [8, 14, 21, 25, 43, 48, 49, 56, 63, 68, 73, 81,
          88, 94, 99, 107, 113, 119, 120, 125, 126, 127, 128]
KEEP_E = [i for i in range(1, 129) if i not in DROP_E]  # 1-based
KEEP_IDX = np.array(KEEP_E, dtype=np.int64) - 1         # 0-based
assert len(KEEP_E) == 105

# -----------------------
# 3) Segmentation
# -----------------------
def segment_signal(data, fs=FS, win_sec=WIN_SEC, step_sec=STEP_SEC, max_sec=MAX_SEC):
    win = int(win_sec * fs)
    step = int(step_sec * fs)
    if max_sec is None:
        max_samples = data.shape[1]
    else:
        max_samples = min(data.shape[1], int(max_sec * fs))
    segments = []
    for start in range(0, max_samples - win + 1, step):
        end = start + win
        segments.append(data[:, start:end])
    return segments

# -----------------------
# 4) Node features: Hjorth + PSD
# -----------------------
def hjorth_features(seg):
    x = seg.astype(np.float32)
    activity = np.var(x, axis=1)
    dx = np.diff(x, axis=1)
    var_dx = np.var(dx, axis=1) + 1e-8
    mobility = np.sqrt(var_dx / (activity + 1e-8))
    ddx = np.diff(dx, axis=1)
    var_ddx = np.var(ddx, axis=1) + 1e-8
    mobility_dx = np.sqrt(var_ddx / var_dx)
    complexity = mobility_dx / (mobility + 1e-8)
    return activity, mobility, complexity

def psd_welch_total(seg, fs=FS, fmin=PSD_FMIN, fmax=PSD_FMAX):
    x = seg.astype(np.float32)
    n_ch = x.shape[0]
    out = np.zeros(n_ch, dtype=np.float32)
    for ch in range(n_ch):
        freqs, pxx = welch(x[ch], fs=fs, nperseg=min(len(x[ch]), int(fs * 2)))
        m = (freqs >= fmin) & (freqs <= fmax)
        out[ch] = float(np.sum(pxx[m]))
    return out

def ar_yule_walker_coeffs(x, order):
    x = x.astype(np.float64)
    x = x - x.mean()
    r = np.array([np.dot(x[k:], x[:len(x)-k]) for k in range(order+1)], dtype=np.float64) / len(x)
    R = np.empty((order, order), dtype=np.float64)
    for i in range(order):
        for j in range(order):
            R[i, j] = r[abs(i-j)]
    rhs = r[1:]
    try:
        a = np.linalg.solve(R + 1e-8*np.eye(order), rhs)
    except np.linalg.LinAlgError:
        a = np.linalg.lstsq(R + 1e-8*np.eye(order), rhs, rcond=None)[0]
    sigma2 = max(r[0] - float(np.dot(a, rhs)), 1e-8)
    return a, sigma2

def psd_ar_total(seg, fs=FS, order=AR_ORDER, n_freq=PSD_NFREQ, fmin=PSD_FMIN, fmax=PSD_FMAX):
    x = seg.astype(np.float32)
    n_ch, T = x.shape
    out = np.zeros(n_ch, dtype=np.float32)
    w = np.linspace(0.0, np.pi, n_freq, dtype=np.float64)
    freqs = w * fs / (2.0 * np.pi)
    m = (freqs >= fmin) & (freqs <= fmax)
    k = np.arange(1, order+1, dtype=np.float64)
    for ch in range(n_ch):
        a, sigma2 = ar_yule_walker_coeffs(x[ch], order)
        ex = np.exp(-1j * np.outer(w, k))
        denom = np.abs(1.0 + ex @ a)**2
        pxx = sigma2 / denom
        out[ch] = float(np.sum(pxx[m]))
    return out.astype(np.float32)

def compute_node_features(seg):
    activity, mobility, complexity = hjorth_features(seg)
    if PSD_METHOD.lower() == "ar":
        psd = psd_ar_total(seg)
    else:
        psd = psd_welch_total(seg)
    feats = np.stack([activity, mobility, complexity, psd], axis=1).astype(np.float32)
    return feats  # (N,4)

# -----------------------
# 5) Adjacency: abs Pearson correlation -> D^{-1/2} A D^{-1/2}
# -----------------------
def normalized_abs_corr(seg):
    corr = np.corrcoef(seg)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    A = np.abs(corr).astype(np.float32)
    d = np.sum(A, axis=1)
    d_inv_sqrt = np.power(d + 1e-8, -0.5)
    D_inv_sqrt = np.diag(d_inv_sqrt.astype(np.float32))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return A_norm.astype(np.float32)

# -----------------------
# 6) Build subject cache (segment-level items; subject-wise split)
# -----------------------
def group_mat_files_by_subject(mat_files):
    g = {}
    for p in mat_files:
        sid = parse_subject_id(p)
        g.setdefault(sid, []).append(p)
    for sid in g:
        g[sid] = sorted(g[sid])
    return g

def build_subject_cache(mat_files, label_map=None):
    if GROUP_BY_SUBJECT_ID:
        groups = group_mat_files_by_subject(mat_files)
        subject_ids = sorted(groups.keys())
        file_lists = [groups[sid] for sid in subject_ids]
    else:
        subject_ids = [parse_subject_id(p) for p in mat_files]
        file_lists = [[p] for p in mat_files]

    labels = []
    subj_X, subj_A = [], []

    # Warnings about label heuristics
    if label_map is None:
        print("WARNING: LABEL_CSV is not provided. Labels will be inferred from file names / heuristics.")
        print("         For a reproducibility study, providing LABEL_CSV is strongly recommended.")

    for sid, paths in tqdm(list(zip(subject_ids, file_lists)), desc="Loading subjects"):
        y = parse_label(paths[0], sid, label_map=label_map)

        # load and (optionally) concat multiple files for this subject
        eeg_list = []
        for p in paths:
            eeg = load_eeg_mat(p)  # (128,T)
            eeg = eeg[KEEP_IDX, :]  # (105,T)
            if APPLY_BANDPASS:
                eeg = bandpass_fir(eeg, fs=FS)
            eeg_list.append(eeg)

        if CONCAT_FILES_PER_SUBJECT and len(eeg_list) > 1:
            eeg = np.concatenate(eeg_list, axis=1)
        else:
            eeg = eeg_list[0]

        segments = segment_signal(eeg, fs=FS)
        if len(segments) == 0:
            print(f"Warning: no segments for {sid}, skip")
            continue

        if ADJ_PER_SEGMENT:
            X = np.stack([compute_node_features(seg) for seg in segments], axis=0).astype(np.float32)
            A = np.stack([normalized_abs_corr(seg) for seg in segments], axis=0).astype(np.float16)  # save RAM
        else:
            X = np.stack([compute_node_features(seg) for seg in segments], axis=0).astype(np.float32)
            A0 = normalized_abs_corr(eeg[:, :min(eeg.shape[1], int(MAX_SEC*FS))])
            A = np.repeat(A0[None, :, :], repeats=X.shape[0], axis=0).astype(np.float16)

        labels.append(int(y))
        subj_X.append(X)
        subj_A.append(A)

    return subject_ids, np.array(labels, dtype=np.int64), subj_X, subj_A

def sanity_report(subject_ids, subj_y, subj_X):
    n_sub = len(subject_ids)
    seg_counts = np.array([x.shape[0] for x in subj_X], dtype=np.int64)
    print("\n=== Sanity report ===")
    print("Subjects:", n_sub)
    print("Class counts (label=1 MDD):", int(subj_y.sum()), " / label=0:", int((subj_y==0).sum()))
    print("Segments per subject: min/median/max =", int(seg_counts.min()), int(np.median(seg_counts)), int(seg_counts.max()))
    # Check for duplicate subject IDs (should not happen when grouping)
    print("Unique subject IDs:", len(set(subject_ids)))
    # Note: paper reports 393 segments/subject when MAX_SEC=200; 353 when MAX_SEC=180
    expected = int((MAX_SEC - WIN_SEC) / STEP_SEC) + 1
    print("Expected seg/subject given MAX_SEC:", expected, f"(MAX_SEC={MAX_SEC}, WIN={WIN_SEC}, STEP={STEP_SEC})")

# -----------------------
# 7) Dataset
# -----------------------
class GraphSegmentDataset(Dataset):
    def __init__(self, subject_indices, subj_X, subj_A, subj_y, return_sub_idx=False, feat_mean=None, feat_std=None):
        self.subj_X = subj_X
        self.subj_A = subj_A
        self.subj_y = subj_y
        self.return_sub_idx = return_sub_idx
        self.feat_mean = feat_mean
        self.feat_std = feat_std
        self.samples = []
        for s in subject_indices:
            n_seg = subj_X[s].shape[0]
            for k in range(n_seg):
                self.samples.append((int(s), int(k)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s, k = self.samples[idx]
        x = self.subj_X[s][k]                    # (N,4)
        if self.feat_mean is not None and self.feat_std is not None:
            x = (x - self.feat_mean) / self.feat_std
        a = self.subj_A[s][k].astype(np.float32) # (N,N)
        y = float(self.subj_y[s])
        if self.return_sub_idx:
            return torch.from_numpy(x), torch.from_numpy(a), torch.tensor(y, dtype=torch.float32), torch.tensor(s, dtype=torch.long)
        return torch.from_numpy(x), torch.from_numpy(a), torch.tensor(y, dtype=torch.float32)

def compute_feature_stats(train_subject_indices, subj_X):
    """Compute feature-wise mean/std from training subjects only (no test leakage)."""
    xs = []
    for s in train_subject_indices:
        x = subj_X[int(s)]  # (n_seg, N, 4)
        xs.append(x.reshape(-1, x.shape[-1]))
    arr = np.concatenate(xs, axis=0)
    mean = arr.mean(axis=0).astype(np.float32)
    std = (arr.std(axis=0) + 1e-8).astype(np.float32)
    # reshape to broadcast over (N,4)
    return mean.reshape(1, -1), std.reshape(1, -1)

# -----------------------
# 8) Model: GICN (input layer with learnable W_alpha)
# -----------------------
class GraphInputLayer(nn.Module):
    def __init__(self, num_nodes, in_channels, out_channels):
        super().__init__()
        self.W_alpha = nn.Parameter(torch.rand(num_nodes, num_nodes))
        self.linear = nn.Linear(in_channels, out_channels, bias=False)

    def forward(self, x, adj_norm):
        # x: [B,N,Fin], adj_norm: [B,N,N]
        A_eff = adj_norm * self.W_alpha  # broadcast over batch
        h = torch.bmm(A_eff, x)
        return self.linear(h)

class GraphConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels, bias=False)

    def forward(self, x, adj_norm):
        h = torch.bmm(adj_norm, x)
        return self.linear(h)

class GICN(nn.Module):
    def __init__(self, num_nodes=105, in_channels=4, dense_dim=6, dropout=0.2):
        super().__init__()
        self.input_layer = GraphInputLayer(num_nodes, in_channels, 4)
        self.bn1 = nn.BatchNorm1d(4)

        self.gcn2 = GraphConvLayer(4, 8)
        self.bn2 = nn.BatchNorm1d(8)

        self.gcn3 = GraphConvLayer(8, 16)
        self.bn3 = nn.BatchNorm1d(16)

        self.act = nn.LeakyReLU(0.2)
        self.fc1 = nn.Linear(num_nodes * 16, dense_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(dense_dim, 1)

    def forward(self, x, adj_norm):
        h = self.act(self.input_layer(x, adj_norm))
        h = self.bn1(h.permute(0,2,1)).permute(0,2,1)

        h = self.act(self.gcn2(h, adj_norm))
        h = self.bn2(h.permute(0,2,1)).permute(0,2,1)

        h = self.act(self.gcn3(h, adj_norm))
        h = self.bn3(h.permute(0,2,1)).permute(0,2,1)

        B, N, F = h.shape
        h = h.reshape(B, N*F)
        h = self.act(self.fc1(h))
        h = self.dropout(h)
        return self.fc_out(h).squeeze(-1)  # logits

# -----------------------
# 9) Train / eval
# -----------------------

@torch.no_grad()
def eval_loader(model, loader, device, subject_majority=False):
    model.eval()
    all_logits, all_y = [], []
    all_sub = []
    for batch in loader:
        if len(batch) == 4:
            xb, ab, yb, sb = batch
        else:
            xb, ab, yb = batch
            sb = None

        xb = xb.to(device)
        ab = ab.to(device)
        yb = yb.to(device)

        logits = model(xb, ab)
        all_logits.append(logits.detach().cpu())
        all_y.append(yb.detach().cpu())
        if sb is not None:
            all_sub.append(sb.detach().cpu())

    logits = torch.cat(all_logits).numpy()
    y = torch.cat(all_y).numpy().astype(np.int64)
    probs = 1.0 / (1.0 + np.exp(-logits))

    if not subject_majority:
        pred = (probs >= 0.5).astype(np.int64)
        acc = accuracy_score(y, pred)
        rec = recall_score(y, pred, zero_division=0)
        prec = precision_score(y, pred, zero_division=0)
        try:
            auc = roc_auc_score(y, probs)
        except ValueError:
            auc = float("nan")
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
        spec = tn / (tn + fp + 1e-8)
        return {"acc": acc, "auc": auc, "recall": rec, "spec": spec, "prec": prec}

    # Subject-majority / mean-prob evaluation
    subs = torch.cat(all_sub).numpy().astype(np.int64)
    subj_prob = {}
    subj_y = {}
    for p, yy, s in zip(probs, y, subs):
        subj_prob.setdefault(int(s), []).append(float(p))
        subj_y[int(s)] = int(round(float(yy)))

    agg_probs = []
    agg_y = []
    for s in sorted(subj_prob.keys()):
        agg_probs.append(np.mean(subj_prob[s]))
        agg_y.append(subj_y[s])

    agg_probs = np.array(agg_probs, dtype=np.float64)
    agg_y = np.array(agg_y, dtype=np.int64)
    agg_pred = (agg_probs >= 0.5).astype(np.int64)

    acc = accuracy_score(agg_y, agg_pred)
    rec = recall_score(agg_y, agg_pred, zero_division=0)
    prec = precision_score(agg_y, agg_pred, zero_division=0)
    try:
        auc = roc_auc_score(agg_y, agg_probs)
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(agg_y, agg_pred, labels=[0,1]).ravel()
    spec = tn / (tn + fp + 1e-8)
    return {"acc": acc, "auc": auc, "recall": rec, "spec": spec, "prec": prec}


def train_one_fold(train_sub_idx, test_sub_idx, subj_X, subj_A, subj_y, device):
    # subject-wise validation split inside train
    if VAL_RATIO and VAL_RATIO > 0.0:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=SEED)
        tr_idx, va_idx = next(sss.split(train_sub_idx, subj_y[train_sub_idx]))
        tr_sub = train_sub_idx[tr_idx]
        va_sub = train_sub_idx[va_idx]
    else:
        tr_sub = train_sub_idx
        va_sub = None

    feat_mean, feat_std = (None, None)
    if NORMALIZE_NODE_FEATURES:
        feat_mean, feat_std = compute_feature_stats(tr_sub, subj_X)

    train_ds = GraphSegmentDataset(tr_sub, subj_X, subj_A, subj_y, return_sub_idx=False, feat_mean=feat_mean, feat_std=feat_std)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

    if va_sub is not None:
        val_ds = GraphSegmentDataset(va_sub, subj_X, subj_A, subj_y, return_sub_idx=False, feat_mean=feat_mean, feat_std=feat_std)
        val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    else:
        val_ld = None

    test_ds = GraphSegmentDataset(test_sub_idx, subj_X, subj_A, subj_y, return_sub_idx=EVAL_SUBJECT_MAJORITY, feat_mean=feat_mean, feat_std=feat_std)
    test_ld = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    num_nodes = subj_X[0].shape[1]
    model = GICN(num_nodes=num_nodes, in_channels=4, dense_dim=DENSE_DIM, dropout=DROPOUT).to(device)
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=BASE_LR)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_STEP, gamma=LR_GAMMA)

    best_state = None
    best_val = -1.0

    for ep in range(1, NUM_EPOCHS+1):
        model.train()
        run_loss = 0.0
        for batch in train_ld:
            xb, ab, yb = batch
            xb = xb.to(device)
            ab = ab.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            logits = model(xb, ab)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            run_loss += loss.item() * xb.size(0)

        sch.step()
        avg_loss = run_loss / max(1, len(train_ds))

        if val_ld is not None:
            m_val = eval_loader(model, val_ld, device, subject_majority=False)
            if m_val["acc"] > best_val:
                best_val = m_val["acc"]
                best_state = copy.deepcopy(model.state_dict())
        else:
            best_state = copy.deepcopy(model.state_dict())

        if ep == 1 or ep % 10 == 0:
            msg = f"Epoch {ep:03d} loss={avg_loss:.4f}"
            if val_ld is not None:
                msg += f" val_acc={m_val['acc']*100:.2f}%"
            print(msg)

    if best_state is not None:
        model.load_state_dict(best_state)

    m_test = eval_loader(model, test_ld, device, subject_majority=EVAL_SUBJECT_MAJORITY)
    return m_test

def summarize_metrics(fold_metrics, key, as_percent=False):
    vals = np.array([m[key] for m in fold_metrics], dtype=np.float64)
    mean = vals.mean()
    std = vals.std()
    if as_percent:
        return mean*100.0, std*100.0
    return mean, std

# -----------------------
# 10) Main
# -----------------------
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    ensure_unzipped(ZIP_PATH, DATA_ROOT)
    mat_files = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.mat"), recursive=True))
    print("mat files:", len(mat_files))

    label_map = load_label_map(LABEL_CSV) if LABEL_CSV else None

    subject_ids, subj_y, subj_X, subj_A = build_subject_cache(mat_files, label_map=label_map)
    print("Loaded subjects:", len(subject_ids))
    sanity_report(subject_ids, subj_y, subj_X)

    # 10-fold subject-wise stratified CV
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    indices = np.arange(len(subject_ids))

    fold_metrics = []
    for fold, (tr, te) in enumerate(skf.split(indices, subj_y), 1):
        print(f"\n=== Fold {fold}/{N_SPLITS} ===")
        m = train_one_fold(tr, te, subj_X, subj_A, subj_y, device)
        print(f"Fold-{fold} test: "
              f"Acc={m['acc']*100:.2f}% AUC={m['auc']:.4f} "
              f"Recall={m['recall']*100:.2f}% Spec={m['spec']*100:.2f}% "
              f"Prec={m['prec']*100:.2f}%")
        fold_metrics.append(m)

    mean_acc, std_acc = summarize_metrics(fold_metrics, "acc", as_percent=True)
    mean_auc, std_auc = summarize_metrics(fold_metrics, "auc", as_percent=False)
    mean_rec, std_rec = summarize_metrics(fold_metrics, "recall", as_percent=True)
    mean_spec, std_spec = summarize_metrics(fold_metrics, "spec", as_percent=True)
    mean_prec, std_prec = summarize_metrics(fold_metrics, "prec", as_percent=True)

    print("\n=== 10-fold subject-wise CV ===")
    print("Evaluation mode:", "subject-majority" if EVAL_SUBJECT_MAJORITY else "segment-level")
    print(f"Accuracy   : {mean_acc:.2f}% ± {std_acc:.2f}%")
    print(f"AUC        : {mean_auc:.4f} ± {std_auc:.4f}")
    print(f"Recall     : {mean_rec:.2f}% ± {std_rec:.2f}%")
    print(f"Specificity: {mean_spec:.2f}% ± {std_spec:.2f}%")
    print(f"Precision  : {mean_prec:.2f}% ± {std_prec:.2f}%")

if __name__ == "__main__":
    main()

Device: cuda
Extracting zip... (first run only)
Done. Found 53 .mat files under /content/modma_lanzhou.
mat files: 53
         For a reproducibility study, providing LABEL_CSV is strongly recommended.


Loading subjects:   0%|          | 0/53 [00:00<?, ?it/s]

Loaded subjects: 53

=== Sanity report ===
Subjects: 53
Class counts (label=1 MDD): 24  / label=0: 29
Segments per subject: min/median/max = 393 393 393
Unique subject IDs: 53
Expected seg/subject given MAX_SEC: 393 (MAX_SEC=200.0, WIN=4.0, STEP=0.5)

=== Fold 1/10 ===
Epoch 001 loss=0.6742
Epoch 010 loss=0.0223
Epoch 020 loss=0.0124
Epoch 030 loss=0.0072
Epoch 040 loss=0.0004
Epoch 050 loss=0.0011
Epoch 060 loss=0.0003


/tmp/ipython-input-4271429604.py:510: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-logits))


Fold-1 test: Acc=70.40% AUC=0.7109 Recall=57.68% Spec=83.12% Prec=77.36%

=== Fold 2/10 ===
Epoch 001 loss=0.6564
Epoch 010 loss=0.0124
Epoch 020 loss=0.0059
Epoch 030 loss=0.0055
Epoch 040 loss=0.0011
Epoch 050 loss=0.0007
Epoch 060 loss=0.0006
Fold-2 test: Acc=66.20% AUC=0.6543 Recall=53.60% Spec=78.80% Prec=71.66%

=== Fold 3/10 ===


...

/tmp/ipython-input-4271429604.py:510: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-logits))


Epoch 001 loss=0.6724
Epoch 010 loss=0.0266
Epoch 020 loss=0.0249
Epoch 030 loss=0.0188
Epoch 040 loss=0.0024
Epoch 050 loss=0.0015
Epoch 060 loss=0.0016
Fold-9 test: Acc=42.39% AUC=0.4331 Recall=47.33% Spec=39.10% Prec=34.13%

=== Fold 10/10 ===


/tmp/ipython-input-4271429604.py:510: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-logits))


Epoch 001 loss=0.6841
Epoch 010 loss=0.0231
Epoch 020 loss=0.0251
Epoch 030 loss=0.0208
Epoch 040 loss=0.0020
Epoch 050 loss=0.0007
Epoch 060 loss=0.0003
Fold-10 test: Acc=47.02% AUC=0.4688 Recall=29.39% Spec=58.78% Prec=32.22%

=== 10-fold subject-wise CV ===
Evaluation mode: segment-level
Accuracy   : 54.57% ± 13.77%
AUC        : 0.5141 ± 0.1602
Recall     : 39.68% ± 25.25%
Specificity: 65.22% ± 18.91%
Precision  : 47.95% ± 20.25%


/tmp/ipython-input-4271429604.py:510: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-logits))


# SSPA‑GCN (Faithful)

In [ ]:
# -*- coding: utf-8 -*-
"""
SSPA‑GCN (MODMA) paper-aligned re-implementation (single-file, LOSO)

This script is based strictly on the SSPA‑GCN paper description, with explicit switches for
paper-ambiguous parts (SSP protocol, adjacency diagonal, MMD bandwidth, SSP merge rule, GRL schedule, pooling).
Use these switches to *quantify* how much performance depends on unspecified details.

Paper reference: "A novel EEG-based graph convolution network for depression detection: Incorporating secondary subject
partitioning and attention mechanism" (Zhang et al., Expert Systems With Applications, 2024).

NOTES (important for reproducibility studies):
- The paper describes SSP as a pre-training step; it does not explicitly state whether SSP is recomputed per LOSO fold.
  This script supports both:
    * SSP_MODE="paper": SSP is computed once using ALL subjects (matches the paper's wording; may leak test distribution
      into domain labels).
    * SSP_MODE="strict": SSP is fitted on train subjects per fold (leak-free).
- Adjacency diagonal is not explicitly discussed. Many GCN pipelines set A_ii = 0 because Chebyshev T0=I already
  provides a self-term. This script supports both via ZERO_DIAG.
- Gaussian kernel bandwidth ξ for MMD is not specified in the paper. This script supports "median" heuristic or a fixed
  float.
"""

import os, glob, math, random, subprocess
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.io import loadmat
from scipy.signal import butter, filtfilt


# ======================
# Config
# ======================
@dataclass
class CFG:
    # Data
    FS: int = 250
    SEG_SECONDS: int = 2
    N_SEGMENTS: int = 150
    N_CHANNELS: int = 128
    BANDS: Tuple[Tuple[float, float], ...] = ((0.5, 4), (4, 8), (8, 12), (12, 35), (35, 100))

    # Paper hyperparams (Table 4, MODMA side)
    PHQ_MAX: float = 27.0
    PHI: float = 0.3
    LR: float = 0.001
    EPOCHS: int = 50
    BATCH_SIZE: int = 300
    ALPHA_L1: float = 0.002
    BETA_L2: float = 0.0125
    DROPOUT: float = 0.5
    DN: int = 12  # MODMA best reported in the paper (Fig.10)

    # Chebyshev GCN (paper does not fully specify -> configurable)
    CHEB_K: int = 3
    GCN_HIDDEN: int = 64
    GCN_OUT: int = 128

    # Pooling (paper does not specify -> configurable)
    # "mean": global average pooling over channels (C dimension)
    # "flatten": concatenate all node features (C*Fout) then MLP
    POOLING: str = "mean"  # "mean" or "flatten"

    # ===== SSP protocol switch =====
    # "paper": SSP once on ALL subjects before LOSO training (paper-like).
    # "strict": per-fold SSP trained on train subjects only (leak-free).
    SSP_MODE: str = "paper"  # "paper" or "strict"

    # SSP / MMD
    MMD_BANDWIDTH: str = "median"  # "median" or float-string e.g. "1.0"
    MMD_ESTIMATOR: str = "biased"  # "biased" stable; "unbiased" can be noisy
    SIGMA_MEDIAN_MAX_SAMPLES: int = 1200

    # SSP merge rule
    # Paper says "averaged"; ambiguous for unequal cluster sizes.
    # False matches literal (fi+fj)/2; True yields true cluster-size mean (order independent).
    WEIGHTED_MERGE: bool = False

    # Feature normalization (paper doesn't specify -> OFF by default)
    ZSCORE_DE_FOR_MODEL: bool = False
    ZSCORE_DE_FOR_SSP: bool = False

    # Adjacency diagonal
    # Many pipelines use ZERO_DIAG=True (A_ii=0) because Chebyshev T0=I already adds self-term.
    ZERO_DIAG: bool = True

    # GRL lambda schedule
    GRL_MODE: str = "dann"  # "constant" or "dann"
    GRL_LAMBDA: float = 1.0

    # Domain loss scalar
    DOMAIN_LOSS_WEIGHT: float = 1.0

    # Regularization target
    REG_WEIGHTS_ONLY: bool = False

    # Training stability
    CLIP_GRAD_NORM: float = 0.0

    # Reproducibility
    SEED: int = 0

    # Data slice strategy (paper implies 150 * 2s = 5 min; dataset recordings may be >= 5 min)
    # "first": use the first 150 segments
    # "random": pick a random contiguous 5-min crop (seeded)
    CROP_MODE: str = "first"  # "first" or "random"


cfg = CFG()


# ======================
# Reproducibility
# ======================
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] device:", device)


# ======================
# Colab unzip (optional)
# ======================
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')  # type: ignore

ZIP_PATH = "/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015.zip"
RAW_ROOT = "/content/modma_raw"
os.makedirs(RAW_ROOT, exist_ok=True)


def unzip_if_needed(zip_path: str, dst_root: str):
    has_mat = any(p.endswith(".mat") for p in glob.glob(os.path.join(dst_root, "**", "*.mat"), recursive=True))
    if has_mat:
        print("[INFO] Found .mat files under", dst_root, "-> skip unzip")
        return
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"ZIP not found: {zip_path}")
    print("[INFO] Unzipping MODMA zip (only once)...")
    subprocess.run(["unzip", "-o", "-qq", zip_path, "-d", dst_root], check=True)
    print("[INFO] Unzip done.")


if IN_COLAB:
    unzip_if_needed(ZIP_PATH, RAW_ROOT)


# ======================
# PHQ-9 map
# ======================
raise RuntimeError(
    "PHQ_SCORES (subject_id -> PHQ-9) was present in the internal run, but is omitted "
    "in this public release to comply with the MODMA Dataset EULA."
)


def is_mdd(subj_id: str) -> int:
    # assumption: IDs starting with "0201" are MDD in this MODMA split
    return 1 if subj_id.startswith("0201") else 0


n_mdd = sum(is_mdd(s) for s in SUBJECT_IDS)
n_hc = len(SUBJECT_IDS) - n_mdd
print(f"[INFO] subjects={len(SUBJECT_IDS)} (MDD={n_mdd}, HC={n_hc})")


# ======================
# Load .mat (robust key selection)
# ======================
def load_eeg_mat(path: str, n_channels: int = 128) -> np.ndarray:
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    keys = [k for k in mat.keys() if not k.startswith("__")]

    candidates = []
    for k in keys:
        v = mat[k]
        try:
            arr = np.asarray(v)
        except Exception:
            continue
        if arr.ndim != 2:
            continue
        if not np.issubdtype(arr.dtype, np.number):
            continue
        # prefer large 2D numeric arrays
        candidates.append((k, arr))

    if not candidates:
        raise RuntimeError(f"No 2D numeric EEG-like array found in {path} keys={keys}")

    k_best, data = max(candidates, key=lambda kv: kv[1].size)
    data = np.asarray(data, dtype=np.float32)

    # normalize orientation to (C,T)
    if data.shape[0] == n_channels + 1:
        data = data[:n_channels, :]
    elif data.shape[1] == n_channels + 1:
        data = data.T[:n_channels, :]
    elif data.shape[0] == n_channels:
        pass
    elif data.shape[1] == n_channels:
        data = data.T
    else:
        # last resort
        if data.shape[0] > data.shape[1]:
            data = data[:n_channels, :]
        else:
            data = data.T[:n_channels, :]
    return data  # (C,T)


# ======================
# Locate .mat files
# ======================
all_mat_files = glob.glob(os.path.join(RAW_ROOT, "**", "*.mat"), recursive=True)
print("[INFO] Found", len(all_mat_files), ".mat files under", RAW_ROOT)


def pick_best_match(paths: List[str]) -> str:
    # If multiple matches per subject, choose largest file size (heuristic).
    if len(paths) == 1:
        return paths[0]
    paths = sorted(paths, key=lambda p: os.path.getsize(p), reverse=True)
    return paths[0]


subj_to_path: Dict[str, str] = {}
for sid in SUBJECT_IDS:
    matches = [p for p in all_mat_files if sid in os.path.basename(p)]
    if not matches:
        raise RuntimeError(f"Could not find .mat for subject {sid}")
    subj_to_path[sid] = pick_best_match(matches)
print("[INFO] Mapped", len(subj_to_path), "subjects to .mat files.")


# ======================
# DE features
# ======================
SEG_LEN = cfg.FS * cfg.SEG_SECONDS
BANDS = list(cfg.BANDS)
N_BANDS = len(BANDS)


def design_band_filters(fs: int = cfg.FS):
    nyq = fs / 2.0
    filters = []
    for low, high in BANDS:
        b, a = butter(4, [low / nyq, high / nyq], btype="bandpass")
        filters.append((b, a))
    return filters


BAND_FILTERS = design_band_filters()


def zscore_de_subjectwise(de: np.ndarray) -> np.ndarray:
    # subject-wise z-score over time windows (axis 0)
    mu = de.mean(axis=0, keepdims=True)
    sd = de.std(axis=0, keepdims=True) + 1e-6
    return (de - mu) / sd


def compute_de_for_subject(eeg: np.ndarray) -> np.ndarray:
    """
    eeg: (C,T)
    returns: (150,C,5)
    """
    C, T = eeg.shape
    max_n_seg = T // SEG_LEN
    if max_n_seg < 1:
        raise RuntimeError(f"Not enough length: T={T}")

    # Choose exactly 150 segments (5 min). If recording is longer, either take first or random crop.
    if max_n_seg >= cfg.N_SEGMENTS:
        if cfg.CROP_MODE == "first":
            start_seg = 0
        else:
            rng = np.random.RandomState(cfg.SEED)
            start_seg = int(rng.randint(0, max_n_seg - cfg.N_SEGMENTS + 1))
        start = start_seg * SEG_LEN
        eeg = eeg[:, start: start + cfg.N_SEGMENTS * SEG_LEN]
        n_seg = cfg.N_SEGMENTS
    else:
        n_seg = max_n_seg
        eeg = eeg[:, : n_seg * SEG_LEN]

    de = np.empty((n_seg, C, N_BANDS), dtype=np.float32)
    for bi, (b, a) in enumerate(BAND_FILTERS):
        filtered = filtfilt(b, a, eeg, axis=1)
        reshaped = filtered.reshape(C, n_seg, SEG_LEN)
        var = reshaped.var(axis=2, ddof=0) + 1e-8
        de_band = 0.5 * np.log(2 * math.pi * math.e * var)  # (C,n_seg)
        de[:, :, bi] = de_band.T  # (n_seg,C)

    # pad if shorter than 150 segments
    if n_seg < cfg.N_SEGMENTS:
        pad = np.repeat(de[-1:], repeats=(cfg.N_SEGMENTS - n_seg), axis=0)
        de = np.concatenate([de, pad], axis=0)
    de = de[:cfg.N_SEGMENTS]
    return de  # (150,C,5)


# ======================
# Adjacency (Eq.2)
# ======================
def compute_adj_from_de(de: np.ndarray, phi: float = cfg.PHI, zero_diag: bool = cfg.ZERO_DIAG) -> np.ndarray:
    """
    de: (O,C,F)
    A_mn = 1 if |Corr(xm,xn)| >= phi else 0
    """
    O, C, F_ = de.shape
    feat = de.transpose(1, 0, 2).reshape(C, -1)  # (C, O*F)
    corr = np.corrcoef(feat)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    A = (np.abs(corr) >= phi).astype(np.float32)
    np.fill_diagonal(A, 0.0 if zero_diag else 1.0)
    return A


# ======================
# Chebyshev polynomials (Eq.4)
# ======================
def compute_chebyshev_polynomials(A: np.ndarray, K: int = cfg.CHEB_K) -> np.ndarray:
    C = A.shape[0]
    D = np.diag(A.sum(axis=1))
    L = D - A
    L = (L + L.T) / 2.0  # enforce symmetry

    eigvals = np.linalg.eigvalsh(L)
    lmax = float(np.max(eigvals).real)
    if lmax < 1e-6:
        L_tilde = np.eye(C, dtype=np.float32)
    else:
        L_tilde = (2.0 * L / lmax) - np.eye(C, dtype=np.float32)

    T_k = [np.eye(C, dtype=np.float32)]
    if K > 1:
        T_k.append(L_tilde.astype(np.float32))
    for k in range(2, K):
        T_k.append(2 * L_tilde @ T_k[-1] - T_k[-2])
    return np.stack(T_k, axis=0)  # (K,C,C)


# ======================
# MMD utilities (Eq.7-8)
# ======================
def pairwise_sq_dists(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    X = X.astype(np.float64, copy=False)
    Y = Y.astype(np.float64, copy=False)
    XX = np.sum(X ** 2, axis=1, keepdims=True)
    YY = np.sum(Y ** 2, axis=1, keepdims=True).T
    d = XX + YY - 2.0 * (X @ Y.T)
    return np.maximum(d, 0.0)


def estimate_sigma_median(all_features: np.ndarray, max_samples: int = 1200, seed: int = 0) -> float:
    rng = np.random.RandomState(seed)
    n_total = all_features.shape[0]
    n = min(max_samples, n_total)
    idx = rng.choice(n_total, size=n, replace=False)
    sub = all_features[idx]
    d = pairwise_sq_dists(sub, sub)
    tri = d[np.triu_indices_from(d, k=1)]
    tri = tri[tri > 0]
    med = np.median(tri) if tri.size > 0 else 1.0
    sigma = math.sqrt(0.5 * med) if med > 0 else 1.0
    return float(sigma)


def parse_bandwidth(bw_cfg) -> float:
    if isinstance(bw_cfg, (int, float)):
        return float(bw_cfg)
    if isinstance(bw_cfg, str):
        s = bw_cfg.strip().lower()
        if s == "median":
            return float("nan")
        return float(s)
    return float("nan")


def mmd_rbf(X: np.ndarray, Y: np.ndarray, sigma: float, estimator: str = "biased") -> float:
    sigma = float(max(sigma, 1e-6))
    gamma = 1.0 / (2.0 * sigma * sigma)

    Kxx = np.exp(-gamma * pairwise_sq_dists(X, X))
    Kyy = np.exp(-gamma * pairwise_sq_dists(Y, Y))
    Kxy = np.exp(-gamma * pairwise_sq_dists(X, Y))

    n = X.shape[0]
    m = Y.shape[0]

    if estimator.lower() == "biased":
        val = float(Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean())
        return max(val, 0.0)

    # unbiased
    if n > 1:
        Kxx2 = Kxx.copy()
        np.fill_diagonal(Kxx2, 0.0)
        term_x = Kxx2.sum() / (n * (n - 1))
    else:
        term_x = 0.0

    if m > 1:
        Kyy2 = Kyy.copy()
        np.fill_diagonal(Kyy2, 0.0)
        term_y = Kyy2.sum() / (m * (m - 1))
    else:
        term_y = 0.0

    term_xy = Kxy.mean()
    val = float(term_x + term_y - 2.0 * term_xy)
    return max(val, 0.0)


# ======================
# SSP (Sec.3.4.2 / Fig.6)
# ======================
def run_ssp_iterative(features_list: List[np.ndarray],
                      num_domains: int,
                      bandwidth_cfg="median",
                      estimator: str = "biased",
                      weighted_merge: bool = False,
                      seed: int = 0):
    """
    features_list: list length S, each (O,D) where O=150 segments
    returns: domain_labels (S,), clusters (list of member lists), sigma_used
    """
    S = len(features_list)
    if S < num_domains:
        raise ValueError(f"SSP: S({S}) < num_domains({num_domains})")

    bw = parse_bandwidth(bandwidth_cfg)
    if np.isnan(bw):
        all_concat = np.concatenate(features_list, axis=0)
        sigma = estimate_sigma_median(all_concat, max_samples=cfg.SIGMA_MEDIAN_MAX_SAMPLES, seed=seed)
    else:
        sigma = float(bw)

    print(f"[SSP] sigma={sigma:.4f}, S={S}, DN={num_domains}, estimator={estimator}, weighted_merge={weighted_merge}")

    cluster_feats = [f.copy() for f in features_list]
    cluster_members = [[i] for i in range(S)]

    n = S
    dist = np.full((n, n), np.inf, dtype=np.float64)
    for i in range(n):
        Xi = cluster_feats[i]
        for j in range(i + 1, n):
            Xj = cluster_feats[j]
            d = mmd_rbf(Xi, Xj, sigma, estimator=estimator)
            dist[i, j] = dist[j, i] = d

    # bottom-up hierarchical clustering until remaining clusters == num_domains
    while len(cluster_feats) > num_domains:
        i, j = np.unravel_index(np.argmin(dist), dist.shape)
        if i == j or not np.isfinite(dist[i, j]):
            print("[SSP] Warning: no finite min distance; break.")
            break

        fi, fj = cluster_feats[i], cluster_feats[j]
        mi, mj = cluster_members[i], cluster_members[j]

        if weighted_merge:
            wi, wj = len(mi), len(mj)
            new_feat = (wi * fi + wj * fj) / float(wi + wj)
        else:
            new_feat = (fi + fj) / 2.0  # literal "average"

        new_members = mi + mj

        keep = [k for k in range(len(cluster_feats)) if k not in (i, j)]
        new_cluster_feats = [cluster_feats[k] for k in keep] + [new_feat]
        new_cluster_members = [cluster_members[k] for k in keep] + [new_members]

        n_new = len(new_cluster_feats)
        new_dist = np.full((n_new, n_new), np.inf, dtype=np.float64)

        old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(keep)}
        for a_old in keep:
            a_new = old_to_new[a_old]
            for b_old in keep:
                b_new = old_to_new[b_old]
                if a_old == b_old:
                    continue
                new_dist[a_new, b_new] = dist[a_old, b_old]

        new_idx = n_new - 1
        Xnew = new_feat
        for k_new in range(n_new - 1):
            Xk = new_cluster_feats[k_new]
            d = mmd_rbf(Xk, Xnew, sigma, estimator=estimator)
            new_dist[k_new, new_idx] = new_dist[new_idx, k_new] = d

        cluster_feats = new_cluster_feats
        cluster_members = new_cluster_members
        dist = new_dist

    domain_labels = np.empty(S, dtype=np.int64)
    for d, members in enumerate(cluster_members):
        for sidx in members:
            domain_labels[sidx] = d
    return domain_labels, cluster_members, sigma


# ======================
# Model
# ======================
class ChebGCNLayer(nn.Module):
    def __init__(self, K: int, in_channels: int, out_channels: int, bias: bool = True):
        super().__init__()
        self.K = K
        self.theta = nn.Parameter(torch.Tensor(K, in_channels, out_channels))
        self.bias = nn.Parameter(torch.Tensor(out_channels)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.theta)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor, cheb: torch.Tensor) -> torch.Tensor:
        """
        x: (B,C,Fin)
        cheb: (K,C,C) or (B,K,C,C)
        """
        B, C, Fin = x.shape
        K, Fin2, Fout = self.theta.shape
        assert K == self.K and Fin2 == Fin

        if cheb.dim() == 3:
            cheb = cheb.unsqueeze(0).expand(B, -1, -1, -1)
        elif cheb.dim() != 4:
            raise ValueError("cheb must be (K,C,C) or (B,K,C,C)")

        out = x.new_zeros((B, C, Fout))
        for k in range(K):
            T_k = cheb[:, k]  # (B,C,C)
            Tx = torch.bmm(T_k, x)  # (B,C,Fin)
            out = out + torch.matmul(Tx, self.theta[k])  # (B,C,Fout)

        if self.bias is not None:
            out = out + self.bias
        return out


class GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd: float):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return GradReverseFn.apply(x, lambd)


class SSPAGCN(nn.Module):
    def __init__(self, n_channels: int = cfg.N_CHANNELS, n_bands: int = N_BANDS, K: int = cfg.CHEB_K,
                 gcn_hidden: int = cfg.GCN_HIDDEN, gcn_out: int = cfg.GCN_OUT, num_domains: int = cfg.DN,
                 dropout: float = cfg.DROPOUT, pooling: str = cfg.POOLING):
        super().__init__()
        self.n_channels = n_channels
        self.gcn_out = gcn_out
        self.pooling = pooling

        # Attention matrix Atten: (C,F)
        self.atten = nn.Parameter(torch.ones(n_channels, n_bands))

        self.gcn1 = ChebGCNLayer(K, n_bands, gcn_hidden)
        self.bn1 = nn.BatchNorm1d(gcn_hidden)

        self.gcn2 = ChebGCNLayer(K, gcn_hidden, gcn_out)
        self.bn2 = nn.BatchNorm1d(gcn_out)

        self.dropout = nn.Dropout(dropout)

        feat_dim = gcn_out if pooling == "mean" else gcn_out * n_channels

        self.fc_label = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)  # [Dep, Nor]
        )

        self.fc_domain = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_domains)
        )

    def forward(self, x: torch.Tensor, cheb: torch.Tensor, lambd: float = 1.0):
        """
        x: (B,C,F)
        cheb: (K,C,C) or (B,K,C,C)
        """
        # Eq.(5)
        x = F.relu(x * self.atten)

        # GCN1
        h = self.gcn1(x, cheb)
        B, C, H = h.shape
        h = self.bn1(h.view(B * C, H))
        h = self.dropout(F.relu(h)).view(B, C, H)

        # GCN2
        h = self.gcn2(h, cheb)
        B, C, Fout = h.shape
        h = self.bn2(h.view(B * C, Fout))
        h = self.dropout(F.relu(h)).view(B, C, Fout)

        if self.pooling == "mean":
            feat = h.mean(dim=1)  # (B,Fout)
        elif self.pooling == "flatten":
            feat = h.reshape(B, C * Fout)  # (B,C*Fout)
        else:
            raise ValueError(f"Unknown pooling: {self.pooling}")

        logp_label = F.log_softmax(self.fc_label(feat), dim=1)
        feat_rev = grad_reverse(feat, lambd)
        logp_domain = F.log_softmax(self.fc_domain(feat_rev), dim=1)
        return logp_label, logp_domain, feat


def l1_l2_regularization(model: nn.Module, alpha: float = cfg.ALPHA_L1, beta: float = cfg.BETA_L2,
                         weights_only: bool = False) -> torch.Tensor:
    """
    Eq.(11): α||w||1 + β||w||2
    """
    l1 = torch.tensor(0., device=device)
    l2_sq = torch.tensor(0., device=device)

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if weights_only and p.ndim <= 1:
            continue
        l1 = l1 + p.abs().sum()
        l2_sq = l2_sq + (p ** 2).sum()

    l2 = torch.sqrt(l2_sq + 1e-12)
    return alpha * l1 + beta * l2


# ======================
# Dataset
# ======================
class EEGSegmentDataset(Dataset):
    def __init__(self, subject_indices, de_list, cheb_list, soft_labels, domain_labels_global):
        self.samples = []
        self.de_list = de_list
        self.cheb_list = cheb_list
        self.soft_labels = soft_labels
        self.domain_labels = domain_labels_global

        for si in subject_indices:
            for ti in range(de_list[si].shape[0]):
                self.samples.append((si, ti))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        si, ti = self.samples[idx]
        x = torch.from_numpy(self.de_list[si][ti]).float()  # (C,F)
        cheb = torch.from_numpy(self.cheb_list[si]).float()  # (K,C,C)
        y_soft = torch.from_numpy(self.soft_labels[si]).float()  # (2,) [Dep,Nor]
        d = int(self.domain_labels[si]) if self.domain_labels[si] >= 0 else 0
        y_hard = int(is_mdd(SUBJECT_IDS[si]))  # 1=MDD, 0=HC
        return x, cheb, y_soft, torch.tensor(d, dtype=torch.long), torch.tensor(y_hard, dtype=torch.long), si


# ======================
# Metrics
# ======================
def compute_metrics_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(np.int64)
    y_pred = np.asarray(y_pred).astype(np.int64)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    rec = tp / max(1, (tp + fn))
    pre = tp / max(1, (tp + fp))
    f1 = 2 * pre * rec / max(1e-8, (pre + rec))
    return acc, rec, pre, f1, (tp, tn, fp, fn)


def grl_lambda(step: int, total_steps: int) -> float:
    if cfg.GRL_MODE == "constant":
        return float(cfg.GRL_LAMBDA)
    p = step / float(max(1, total_steps))
    return float(2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)


# ======================
# Precompute per subject: DE, adjacency/cheb, labels, SSP features
# ======================
print("[INFO] Precomputing DE, adjacency, cheb ...")
de_list, cheb_list, soft_label_list, hard_label_list, mmd_feature_list = [], [], [], [], []

for sid in SUBJECT_IDS:
    eeg = load_eeg_mat(subj_to_path[sid], n_channels=cfg.N_CHANNELS)
    de = compute_de_for_subject(eeg)  # (150,128,5)

    # subject-wise z-score optional
    de_model = zscore_de_subjectwise(de) if cfg.ZSCORE_DE_FOR_MODEL else de
    de_ssp = zscore_de_subjectwise(de) if cfg.ZSCORE_DE_FOR_SSP else de

    # adjacency is defined "based on DE features" in the paper; we tie it to model-side DE by default
    A = compute_adj_from_de(de_model, phi=cfg.PHI, zero_diag=cfg.ZERO_DIAG)
    cheb = compute_chebyshev_polynomials(A, K=cfg.CHEB_K)

    score = float(PHQ_SCORES[sid])
    soft = np.array([score / cfg.PHQ_MAX, 1.0 - score / cfg.PHQ_MAX], dtype=np.float32)  # [Dep,Nor]
    hard = is_mdd(sid)

    de_list.append(de_model.astype(np.float32))
    cheb_list.append(cheb.astype(np.float32))
    soft_label_list.append(soft)
    hard_label_list.append(hard)

    mmd_feature_list.append(de_ssp.reshape(cfg.N_SEGMENTS, -1).astype(np.float32))  # (150, 128*5)

print("[INFO] Done for", len(SUBJECT_IDS), "subjects")


# ======================
# SSP domain labels (paper mode: once before training)
# ======================
num_subjects = len(SUBJECT_IDS)

domain_labels_full = None
if cfg.SSP_MODE == "paper":
    print("[INFO] SSP_MODE=paper: computing SSP on ALL subjects once (before LOSO training).")
    domain_labels_full, clusters_full, sigma_full = run_ssp_iterative(
        mmd_feature_list,
        num_domains=cfg.DN,
        bandwidth_cfg=cfg.MMD_BANDWIDTH,
        estimator=cfg.MMD_ESTIMATOR,
        weighted_merge=cfg.WEIGHTED_MERGE,
        seed=cfg.SEED
    )
    counts = {d: int(np.sum(domain_labels_full == d)) for d in range(cfg.DN)}
    print("[INFO] SSP(all) done. domain counts:", counts)


# ======================
# LOSO
# ======================
y_true_subject = np.array(hard_label_list, dtype=np.int64)
y_pred_subject = np.zeros(num_subjects, dtype=np.int64)
seg_true_all, seg_pred_all = [], []

print("[INFO] LOSO start. SSP_MODE =", cfg.SSP_MODE, "| POOLING =", cfg.POOLING, "| ZERO_DIAG =", cfg.ZERO_DIAG)

# make dataloader shuffling deterministic
torch_gen = torch.Generator()
torch_gen.manual_seed(cfg.SEED)

for test_idx in range(num_subjects):
    test_sid = SUBJECT_IDS[test_idx]
    print(f"\n[LOSO] {test_idx + 1}/{num_subjects} test={test_sid}")

    train_idx = [i for i in range(num_subjects) if i != test_idx]

    # Domain labels per fold
    domain_labels_global = np.full(num_subjects, -1, dtype=np.int64)

    if cfg.SSP_MODE == "strict":
        feats_train = [mmd_feature_list[i] for i in train_idx]
        dl_train, clusters, sigma_used = run_ssp_iterative(
            feats_train,
            num_domains=cfg.DN,
            bandwidth_cfg=cfg.MMD_BANDWIDTH,
            estimator=cfg.MMD_ESTIMATOR,
            weighted_merge=cfg.WEIGHTED_MERGE,
            seed=cfg.SEED
        )
        for local_i, global_i in enumerate(train_idx):
            domain_labels_global[global_i] = int(dl_train[local_i])
        domain_labels_global[test_idx] = 0  # dummy
    else:
        domain_labels_global[:] = domain_labels_full

    train_ds = EEGSegmentDataset(train_idx, de_list, cheb_list, soft_label_list, domain_labels_global)
    test_ds = EEGSegmentDataset([test_idx], de_list, cheb_list, soft_label_list, domain_labels_global)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                              drop_last=False, num_workers=0, generator=torch_gen)
    test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                             drop_last=False, num_workers=0)

    model = SSPAGCN(num_domains=cfg.DN, dropout=cfg.DROPOUT, pooling=cfg.POOLING).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=0.0)

    model.train()
    total_steps = cfg.EPOCHS * max(1, len(train_loader))
    step = 0
    for epoch in range(cfg.EPOCHS):
        epoch_loss = 0.0
        for x, cheb, y_soft, d, y_hard, si in train_loader:
            x = x.to(device)
            cheb = cheb.to(device)
            y_soft = y_soft.to(device)
            d = d.to(device)

            lambd = grl_lambda(step, total_steps)
            step += 1

            optimizer.zero_grad()
            log_pl, log_pd, feat = model(x, cheb, lambd=lambd)

            # Soft-label CE: -sum(y * log p)
            loss_label = -(y_soft * log_pl).sum(dim=1).mean()

            # Domain CE (GRL reverses gradient for feature extractor)
            loss_domain = F.nll_loss(log_pd, d) * float(cfg.DOMAIN_LOSS_WEIGHT)

            # Eq.(11) regularization
            loss_reg = l1_l2_regularization(
                model,
                alpha=cfg.ALPHA_L1,
                beta=cfg.BETA_L2,
                weights_only=cfg.REG_WEIGHTS_ONLY
            )

            loss = loss_label + loss_domain + loss_reg
            loss.backward()

            if cfg.CLIP_GRAD_NORM and cfg.CLIP_GRAD_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD_NORM)

            optimizer.step()
            epoch_loss += float(loss.item())

        if (epoch + 1) % max(1, (cfg.EPOCHS // 5)) == 0:
            print(f"  epoch {epoch + 1:3d}/{cfg.EPOCHS}, loss={epoch_loss / max(1, len(train_loader)):.4f}")

    # test
    model.eval()
    seg_probs, seg_true = [], []
    with torch.no_grad():
        for x, cheb, y_soft, d, y_hard, si in test_loader:
            x = x.to(device)
            cheb = cheb.to(device)
            log_pl, _, _ = model(x, cheb, lambd=0.0)
            probs = log_pl.exp().cpu().numpy()  # (B,2) [Dep,Nor]
            seg_probs.append(probs)
            seg_true.extend(y_hard.numpy().tolist())

    seg_probs = np.concatenate(seg_probs, axis=0)

    # segment-level prediction (for sanity / optional confusion matrix like Fig.8 in the paper)
    seg_pred_class = np.argmax(seg_probs, axis=1)      # 0=Dep, 1=Nor
    seg_pred = (seg_pred_class == 0).astype(np.int64)  # Dep -> 1 (MDD)
    seg_true_all.extend(seg_true)
    seg_pred_all.extend(seg_pred.tolist())

    # subject-level: mean prob over 150 segments (paper states they average segments per subject for evaluation)
    mean_prob = seg_probs.mean(axis=0)  # [Dep,Nor]
    pred_mdd = int(mean_prob[0] >= mean_prob[1])
    y_pred_subject[test_idx] = pred_mdd
    print(f"  subject pred: {'MDD' if pred_mdd == 1 else 'HC'} (Dep={mean_prob[0]:.3f}, Nor={mean_prob[1]:.3f})")

# Final metrics
acc_s, rec_s, pre_s, f1_s, cm_s = compute_metrics_binary(y_true_subject, y_pred_subject)
acc_g, rec_g, pre_g, f1_g, cm_g = compute_metrics_binary(seg_true_all, seg_pred_all)

print("\n===== SUBJECT-LEVEL (mean over 150 segments per subject) =====")
print(f"Accuracy:  {acc_s * 100:.2f} %")
print(f"Recall:    {rec_s * 100:.2f} % (MDD)")
print(f"Precision: {pre_s * 100:.2f} % (MDD)")
print(f"F1-score:  {f1_s * 100:.2f} %")
print("Confusion (TP,TN,FP,FN):", cm_s)

print("\n===== SEGMENT-LEVEL (each 2s segment; optional) =====")
print(f"Accuracy:  {acc_g * 100:.2f} %")
print(f"Recall:    {rec_g * 100:.2f} % (MDD)")
print(f"Precision: {pre_g * 100:.2f} % (MDD)")
print(f"F1-score:  {f1_g * 100:.2f} %")
print("Confusion (TP,TN,FP,FN):", cm_g)

[INFO] device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Found .mat files under /content/modma_raw -> skip unzip
[INFO] subjects=53 (MDD=24, HC=29)
[INFO] Found 53 .mat files under /content/modma_raw
[INFO] Mapped 53 subjects to .mat files.
[INFO] Precomputing DE, adjacency, cheb ...
[INFO] Done for 53 subjects
[INFO] SSP_MODE=paper: computing SSP on ALL subjects once (before LOSO training).
[SSP] sigma=15.8723, S=53, DN=12, estimator=biased, weighted_merge=False
[INFO] SSP(all) done. domain counts: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 23, 11: 20}
[INFO] LOSO start. SSP_MODE = paper | POOLING = mean | ZERO_DIAG = True

[LOSO] 1/53 
  epoch  10/50, loss=4.5337
  epoch  20/50, loss=3.5759
  epoch  30/50, loss=3.0405
  epoch  40/50, loss=2.5785
  epoch  50/50, loss=2.3353
  subject pred: MDD (Dep=0.676, Nor=0.324)

[LOSO] 2/53 
  epoch  10/50, loss=4.6101
  epoch 

# GDN (Faithful)

In [ ]:
# ============================================================
# GDN (Mao et al., 2024) reimplementation for MODMA  [LABEL-AUTO v3]
# - single Colab cell
# - works even if folder names do NOT contain "MDD/HC"
# - label inference via subject-id prefixes (auto or forced)
# ============================================================

import os, re, glob, math, subprocess, random, warnings
from collections import defaultdict, Counter
import itertools

import numpy as np
from scipy.io import loadmat

# ---- Colab / Drive ----
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

# ---- deps ----
try:
    import pywt
    from tqdm import tqdm
except ImportError:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets", "tqdm"], check=True)
    import pywt
    from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# Reproducibility
# ----------------------------
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 0
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| Device:", DEVICE)

# ============================================================
# Paths
# ============================================================
ZIP_PATH  = "/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015.zip"
DATA_ROOT = "/content/modma_raw"
os.makedirs(DATA_ROOT, exist_ok=True)

if os.path.exists(ZIP_PATH):
    print("Unzipping (overwrite, quiet)...")
    subprocess.run(["unzip", "-o", "-q", ZIP_PATH, "-d", DATA_ROOT], check=True)
else:
    print("WARNING: zip not found:", ZIP_PATH)

mat_files = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.mat"), recursive=True))
print("Found .mat:", len(mat_files))
if len(mat_files) == 0:
    raise RuntimeError("No .mat found. Check ZIP_PATH/DATA_ROOT.")

# ============================================================
# Paper-aligned config + ambiguity switches
# ============================================================
FS = 250.0
SEG_LEN = 2500          # 10s * 250Hz
K_SIM = 10              # paper: best when k=10
WAVELET_NAME = "db6"
DWT_MODE = "symmetric"

FORCE_8_SEGS_PER_SUBJECT = True
MAX_SEGS_PER_SUBJECT = 8

# --- ambiguity switches (log these in your reproducibility paper) ---
SIMILARITY_ON_FILTERED = False
INCLUDE_TARGET_IN_ENCODER_INPUT = False
DEMEAN_PER_CHANNEL = False
ZSCORE_PER_CHANNEL = False

# Hamming ambiguity (paper has unclear equation)
# - "numpy_hamming": np.hamming(N)
# - "paper_plus_2pi": 0.54 + 0.46*cos(2πt/(N-1))
# - "paper_minus_2pi": 0.54 - 0.46*cos(2πt/(N-1))  (classic)
# - "paper_literal": 0.54 + 0.46*cos(t)
HAMMING_MODE = "paper_plus_2pi"

USE_ACTIVATION = False

# training
EPOCHS = 120
LR = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 256

# dataloader workers (0 is safest for big python objects)
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# ============================================================
# Subject ID inference (works for filenames like "02010002rest 20150416 1017..mat")
# ============================================================
SUBJECT_PATTERNS = [
    re.compile(r"^\d{3,10}$"),        # pure digits folder
    re.compile(r"^(sub|s)\d{2,10}$", re.IGNORECASE),
    re.compile(r"^(subject)\d{2,10}$", re.IGNORECASE),
]
GENERIC_DIRS = {
    "mdd","hc","control","healthy","normal","depression","patient",
    "eeg","eegs","data","dataset","modma","raw"
}

def infer_subject_id(path: str):
    parts = os.path.normpath(path).split(os.sep)
    for i in range(len(parts)-2, -1, -1):
        name = parts[i]
        low = name.lower()
        if low in GENERIC_DIRS:
            continue
        if len(name) > 32:
            continue
        for pat in SUBJECT_PATTERNS:
            if pat.match(name):
                return name
    # fallback to filename digits: choose the FIRST of the longest digit runs
    fn = os.path.basename(path)
    digs = re.findall(r"\d+", fn)
    if digs:
        maxlen = max(len(x) for x in digs)
        for x in digs:
            if len(x) == maxlen:
                return x
    return os.path.splitext(fn)[0]

def sid_digits(sid: str):
    ds = "".join(re.findall(r"\d+", str(sid)))
    return ds if ds else str(sid)

# ============================================================
# Label inference (THIS FIXES YOUR ERROR)
# ============================================================
# Options:
#  - "auto_prefix": infer labels by grouping subject-id prefixes to best match expected counts (24/29)
#  - "force_prefix": use FORCE_PREFIX_LABEL dict
#  - "none": raise (force you to provide mapping)
LABEL_MODE = "auto_prefix"

EXPECTED_MDD = 24
EXPECTED_HC  = 29

# If LABEL_MODE="force_prefix": map prefix -> label(1=MDD, 0=HC)
# Example: {"0201":1, "0103":0}
FORCE_PREFIX_LEN = 4
FORCE_PREFIX_LABEL = {"0201": 1}  # everything else becomes 0 by default

# prefix lengths to try for auto
AUTO_PREFIX_LENS = [4, 3, 2]
AUTO_MAX_GROUPS = 12  # if too many groups, auto becomes unstable

def infer_label_from_path_hints(path: str):
    # (kept for completeness; your current MODMA folder likely has none)
    low = path.lower()
    if any(h in low for h in ("mdd", "dep", "depress", "patient", "case")):
        return 1
    if any(h in low for h in ("hc", "control", "healthy", "normal")):
        return 0
    return None

def build_prefix_groups(subject_ids, prefix_len):
    groups = defaultdict(list)
    for sid in subject_ids:
        d = sid_digits(sid)
        pref = d[:prefix_len] if len(d) >= prefix_len else d
        groups[pref].append(sid)
    return groups

def choose_auto_prefix_mapping(subject_ids):
    # choose a prefix length that yields small number of groups
    chosen_len = None
    chosen_groups = None
    for L in AUTO_PREFIX_LENS:
        groups = build_prefix_groups(subject_ids, L)
        if 1 < len(groups) <= AUTO_MAX_GROUPS:
            chosen_len = L
            chosen_groups = groups
            break
    if chosen_len is None:
        raise RuntimeError(
            f"AUTO label inference failed: too many unique prefix groups.\n"
            f"Try LABEL_MODE='force_prefix' with FORCE_PREFIX_LABEL, or reduce AUTO_MAX_GROUPS."
        )

    # counts per prefix
    prefixes = sorted(chosen_groups.keys())
    counts = [len(chosen_groups[p]) for p in prefixes]
    total = sum(counts)

    # brute force subset of prefixes to be MDD(=1) to match EXPECTED_MDD
    best = None  # (score, n_groups, mdd_count, subset_bitmask)
    for r in range(1, len(prefixes)):  # non-empty and not all
        for subset in itertools.combinations(range(len(prefixes)), r):
            mdd_count = sum(counts[i] for i in subset)
            # primary: closeness to expected MDD
            score = abs(mdd_count - EXPECTED_MDD)
            # tie-break: prefer fewer prefix groups (simpler rule)
            cand = (score, r, mdd_count, subset)
            if best is None or cand < best:
                best = cand

    if best is None:
        raise RuntimeError("AUTO label inference failed: no subset found (unexpected).")

    score, r, mdd_count, subset = best
    subset = set(subset)

    mapping = {}
    for i, p in enumerate(prefixes):
        mapping[p] = 1 if i in subset else 0

    # report
    print("\n[Label:auto_prefix] selected prefix_len =", chosen_len)
    print("[Label:auto_prefix] prefix groups (prefix -> #subjects):")
    for p in prefixes:
        print(f"  {p} -> {len(chosen_groups[p])} (label={mapping[p]})")
    print(f"[Label:auto_prefix] implied counts: MDD={mdd_count}, HC={total-mdd_count}, target(MDD,HC)=({EXPECTED_MDD},{EXPECTED_HC}), score={score}")
    if score > 3:
        warnings.warn("AUTO label mapping is far from expected counts. Verify prefixes and consider force_prefix.")
    return chosen_len, mapping

# ============================================================
# .mat loading (MODMA shape-priority)
# ============================================================
def _is_numeric_array(x):
    return isinstance(x, np.ndarray) and (np.issubdtype(x.dtype, np.number) or x.dtype == np.object_)

def _score_candidate(arr: np.ndarray):
    if not isinstance(arr, np.ndarray):
        return -1
    shp = arr.shape
    if arr.ndim < 2 or arr.ndim > 4:
        return -1
    score = 0
    if 128 in shp or 129 in shp:
        score += 10
    if 2500 in shp:
        score += 10
    if 8 in shp:
        score += 5
    score += min(5, int(np.log10(arr.size + 1)))
    return score

def load_subject_segments(path: str, seg_len=SEG_LEN, force_n_segs=FORCE_8_SEGS_PER_SUBJECT, max_segs=MAX_SEGS_PER_SUBJECT):
    mat = loadmat(path, squeeze_me=False, struct_as_record=False)
    keys = [k for k in mat.keys() if not k.startswith("__")]

    candidates = []
    for k in keys:
        v = mat[k]
        if _is_numeric_array(v):
            arr = np.asarray(v)
            candidates.append((k, arr, _score_candidate(arr)))
    candidates = sorted(candidates, key=lambda x: x[2], reverse=True)
    if not candidates or candidates[0][2] < 0:
        raise ValueError(f"No usable numeric array in {path}. keys={keys}")

    best_key, data, _ = candidates[0]
    data = np.asarray(data, dtype=np.float32)

    # 2D: (128,T) or (T,128)
    if data.ndim == 2:
        if data.shape[0] in (128,129):
            x = data
        elif data.shape[1] in (128,129):
            x = data.T
        else:
            raise ValueError(f"2D array without 128/129 dim: {data.shape} in {path} (key={best_key})")
        if x.shape[0] == 129:
            x = x[:128]
        T = x.shape[1]
        if T < seg_len:
            return np.zeros((0,128,seg_len), dtype=np.float32)
        n_seg = T // seg_len
        x = x[:, :n_seg*seg_len]
        segs = x.reshape(128, n_seg, seg_len).transpose(1,0,2)

    # 3D: prefer (8,128,2500) permutations
    elif data.ndim == 3:
        shp = data.shape
        axes = [0,1,2]
        ax_seg = next((a for a in axes if shp[a] == 8), None)
        ax_ch  = next((a for a in axes if shp[a] in (128,129)), None)
        ax_t   = next((a for a in axes if shp[a] == seg_len), None)

        if ax_seg is not None and ax_ch is not None and ax_t is not None:
            x = np.moveaxis(data, (ax_seg, ax_ch, ax_t), (0,1,2))
            if x.shape[1] == 129:
                x = x[:, :128, :]
            segs = x.astype(np.float32)
        else:
            # fallback heuristic
            ch_axis = next((ax for ax,sz in enumerate(shp) if sz in (128,129)), None)
            if ch_axis is None:
                raise ValueError(f"3D array without 128/129 dim: {shp} in {path} (key={best_key})")
            other = [a for a in axes if a != ch_axis]
            a1, a2 = other
            if shp[a1] == seg_len:
                time_axis, seg_axis = a1, a2
            elif shp[a2] == seg_len:
                time_axis, seg_axis = a2, a1
            else:
                if shp[a1] >= shp[a2]:
                    time_axis, seg_axis = a1, a2
                else:
                    time_axis, seg_axis = a2, a1

            x = np.moveaxis(data, (seg_axis, ch_axis, time_axis), (0,1,2))
            if x.shape[1] == 129:
                x = x[:, :128, :]
            if x.shape[2] == seg_len:
                segs = x
            else:
                x2 = x.transpose(1,0,2).reshape(128, -1)
                T = x2.shape[1]
                n_seg = T // seg_len
                x2 = x2[:, :n_seg*seg_len]
                segs = x2.reshape(128, n_seg, seg_len).transpose(1,0,2)

    # 4D: flatten
    elif data.ndim == 4:
        shp = data.shape
        ch_axis = next((ax for ax,sz in enumerate(shp) if sz in (128,129)), None)
        if ch_axis is None:
            raise ValueError(f"4D array without 128/129 dim: {shp} in {path} (key={best_key})")
        x = np.moveaxis(data, ch_axis, 1)
        if x.shape[1] == 129:
            x = x[:, :128, :, :]
        d0, ch, d2, d3 = x.shape
        x = x.reshape(d0, ch, d2*d3)
        x2 = x.transpose(1,0,2).reshape(ch, -1)
        T = x2.shape[1]
        n_seg = T // seg_len
        x2 = x2[:, :n_seg*seg_len]
        segs = x2.reshape(ch, n_seg, seg_len).transpose(1,0,2)

    else:
        raise ValueError(f"Unsupported ndim={data.ndim} in {path} (key={best_key})")

    if force_n_segs and segs.shape[0] > max_segs:
        segs = segs[:max_segs]
    return segs.astype(np.float32)

# ============================================================
# Signal processing: cosine sim, FFT bandpass, db6 DWT
# ============================================================
def compute_cosine_topk(eeg_128xT: np.ndarray, k=K_SIM):
    x = eeg_128xT
    norms = np.linalg.norm(x, axis=1, keepdims=True) + 1e-8
    x_norm = x / norms
    sim = x_norm @ x_norm.T
    np.fill_diagonal(sim, -np.inf)
    idx = np.argsort(-sim, axis=1)[:, :k]
    return idx.astype(np.int16)

def make_window(n: int):
    t = np.arange(n, dtype=np.float32)
    if HAMMING_MODE == "numpy_hamming":
        return np.hamming(n).astype(np.float32)
    if HAMMING_MODE == "paper_plus_2pi":
        return (0.54 + 0.46*np.cos(2*np.pi*t/(n-1))).astype(np.float32)
    if HAMMING_MODE == "paper_minus_2pi":
        return (0.54 - 0.46*np.cos(2*np.pi*t/(n-1))).astype(np.float32)
    if HAMMING_MODE == "paper_literal":
        return (0.54 + 0.46*np.cos(t)).astype(np.float32)
    raise ValueError("Unknown HAMMING_MODE")

def bandpass_fft_4_14(eeg_128xT: np.ndarray, fs=FS, low=4.0, high=14.0):
    ch, n = eeg_128xT.shape
    w = make_window(n)
    xw = eeg_128xT * w[None, :]
    X = np.fft.rfft(xw, axis=1)
    freqs = np.fft.rfftfreq(n, d=1.0/fs)
    mask = (freqs >= low) & (freqs <= high)
    X[:, ~mask] = 0.0
    y = np.fft.irfft(X, n=n, axis=1).astype(np.float32)
    y = y / (w[None, :] + 1e-6)
    return y.astype(np.float32)

W = pywt.Wavelet(WAVELET_NAME)
EXPECTED_WLEN = pywt.dwt_coeff_len(SEG_LEN, W.dec_len, mode=DWT_MODE)

def compute_wavelet_coeffs_all(eeg_128xT: np.ndarray, wavelet=WAVELET_NAME):
    cA_list, cD_list = [], []
    for ch in range(eeg_128xT.shape[0]):
        cA, cD = pywt.dwt(eeg_128xT[ch], wavelet, mode=DWT_MODE)
        cA = cA.astype(np.float32); cD = cD.astype(np.float32)
        # force expected length (should be 1255 for SEG_LEN=2500, db6, symmetric)
        if cA.shape[0] != EXPECTED_WLEN:
            if cA.shape[0] > EXPECTED_WLEN:
                s = (cA.shape[0]-EXPECTED_WLEN)//2
                cA = cA[s:s+EXPECTED_WLEN]
                cD = cD[s:s+EXPECTED_WLEN]
            else:
                pad = EXPECTED_WLEN - cA.shape[0]
                cA = np.pad(cA, (0,pad))
                cD = np.pad(cD, (0,pad))
        cA_list.append(cA); cD_list.append(cD)
    return np.stack(cA_list, axis=0), np.stack(cD_list, axis=0)

def idwt_to_len(cA_1d: np.ndarray, cD_1d: np.ndarray, out_len=SEG_LEN, wavelet=WAVELET_NAME):
    rec = pywt.idwt(cA_1d, cD_1d, wavelet, mode=DWT_MODE)
    if rec.shape[0] >= out_len:
        return rec[:out_len].astype(np.float32)
    return np.pad(rec.astype(np.float32), (0, out_len - rec.shape[0]))

# ============================================================
# Segment container
# ============================================================
class SegmentInfo:
    __slots__ = ("index","subject_id","label","cA_all","cD_all","sim_idx")
    def __init__(self, index, subject_id, label, cA_all, cD_all, sim_idx):
        self.index = int(index)
        self.subject_id = str(subject_id)
        self.label = int(label)
        self.cA_all = cA_all
        self.cD_all = cD_all
        self.sim_idx = sim_idx

# ============================================================
# Build subject list
# ============================================================
subj_files = defaultdict(list)
for p in mat_files:
    sid = infer_subject_id(p)
    subj_files[sid].append(p)

subject_ids = list(subj_files.keys())
print("Unique subject IDs:", len(subject_ids))
if len(subject_ids) != len(mat_files):
    warnings.warn("mat_files != unique subject IDs (some subjects have multiple files OR subject_id parsing collided).")

# ============================================================
# Decide subject_label
# ============================================================
subject_label = {}

# 1) use path hints if present (rare in your current folder)
path_hint_votes = defaultdict(list)
for sid, files in subj_files.items():
    for p in files:
        y = infer_label_from_path_hints(p)
        if y is not None:
            path_hint_votes[sid].append(y)
if all(len(v)>0 for v in path_hint_votes.values()) and len(path_hint_votes)==len(subject_ids):
    # all subjects got path hints
    for sid, votes in path_hint_votes.items():
        subject_label[sid] = Counter(votes).most_common(1)[0][0]
    print("[Label] inferred from PATH hints for all subjects.")
else:
    if LABEL_MODE == "none":
        raise RuntimeError("No labels in path, and LABEL_MODE='none'. Provide manual mapping.")
    elif LABEL_MODE == "force_prefix":
        # apply forced prefix rule
        for sid in subject_ids:
            d = sid_digits(sid)
            pref = d[:FORCE_PREFIX_LEN] if len(d) >= FORCE_PREFIX_LEN else d
            subject_label[sid] = int(FORCE_PREFIX_LABEL.get(pref, 0))
        print("\n[Label:force_prefix] prefix_len =", FORCE_PREFIX_LEN)
        print("[Label:force_prefix] FORCE_PREFIX_LABEL =", FORCE_PREFIX_LABEL)
    elif LABEL_MODE == "auto_prefix":
        auto_len, auto_map = choose_auto_prefix_mapping(subject_ids)
        for sid in subject_ids:
            d = sid_digits(sid)
            pref = d[:auto_len] if len(d) >= auto_len else d
            subject_label[sid] = int(auto_map[pref])
    else:
        raise ValueError("Unknown LABEL_MODE")

n_mdd = sum(1 for v in subject_label.values() if v==1)
n_hc  = sum(1 for v in subject_label.values() if v==0)
print(f"[Label] Subjects: total={len(subject_label)} | label1={n_mdd} | label0={n_hc}")
if not (15 <= n_mdd <= 40 and 15 <= n_hc <= 45):
    warnings.warn("Label counts look suspicious. Verify label rule (maybe swapped prefixes).")

# ============================================================
# Subject ordering (paper: 'first 15' per class)
# ============================================================
def _first_int(s):
    m = re.findall(r"\d+", str(s))
    return int(m[0]) if m else 10**18

ordered_mdd = sorted([s for s,y in subject_label.items() if y==1], key=_first_int)
ordered_hc  = sorted([s for s,y in subject_label.items() if y==0], key=_first_int)

# ============================================================
# Precompute segments
# ============================================================
segments = []
subject_to_seg_ids = defaultdict(list)

print("\nPrecomputing: similarity -> FFT(4-14) -> db6 DWT ...")
seg_counter = 0

for sid in tqdm(ordered_mdd + ordered_hc):
    y = subject_label[sid]
    files = sorted(subj_files[sid])

    subj_seg_list = []
    for path in files:
        segs = load_subject_segments(path)
        if segs.shape[0] > 0:
            subj_seg_list.append(segs)

    if len(subj_seg_list) == 0:
        warnings.warn(f"No segments found for subject {sid}")
        continue

    subj_segs = np.concatenate(subj_seg_list, axis=0)  # (n_seg,128,2500)

    if FORCE_8_SEGS_PER_SUBJECT:
        if subj_segs.shape[0] < MAX_SEGS_PER_SUBJECT:
            warnings.warn(f"Subject {sid}: only {subj_segs.shape[0]} segments (expected 8).")
        subj_segs = subj_segs[:MAX_SEGS_PER_SUBJECT]

    for i in range(subj_segs.shape[0]):
        raw = subj_segs[i]  # (128,2500)

        if DEMEAN_PER_CHANNEL:
            raw = raw - raw.mean(axis=1, keepdims=True)

        if ZSCORE_PER_CHANNEL:
            mu = raw.mean(axis=1, keepdims=True)
            sd = raw.std(axis=1, keepdims=True) + 1e-6
            raw = (raw - mu) / sd

        if SIMILARITY_ON_FILTERED:
            f_for_sim = bandpass_fft_4_14(raw)
            sim_idx = compute_cosine_topk(f_for_sim, k=K_SIM)
            eeg_f = f_for_sim
        else:
            sim_idx = compute_cosine_topk(raw, k=K_SIM)
            eeg_f = bandpass_fft_4_14(raw)

        cA_all, cD_all = compute_wavelet_coeffs_all(eeg_f)

        seg = SegmentInfo(
            index=seg_counter,
            subject_id=sid,
            label=y,
            cA_all=cA_all,
            cD_all=cD_all,
            sim_idx=sim_idx
        )
        segments.append(seg)
        subject_to_seg_ids[sid].append(seg_counter)
        seg_counter += 1

print(f"Segments prepared: {len(segments)} | EXPECTED_WLEN={EXPECTED_WLEN}")
if EXPECTED_WLEN != 1255:
    warnings.warn(f"Wavelet coeff len is {EXPECTED_WLEN}, paper reports 1255 for MODMA+db6. Check DWT_MODE/Hamming/filtering.")

# ============================================================
# Subject-wise split (paper: first 15 train, 15-20 val, rest test) per class
# ============================================================
def split_subjects(subj_list):
    return subj_list[:15], subj_list[15:20], subj_list[20:]

mdd_tr, mdd_va, mdd_te = split_subjects(ordered_mdd)
hc_tr,  hc_va,  hc_te  = split_subjects(ordered_hc)

def segs_from_subjects(subj_ids):
    out = []
    for sid in subj_ids:
        out.extend(subject_to_seg_ids.get(sid, []))
    return sorted(out)

mdd_tr_seg = segs_from_subjects(mdd_tr)
mdd_va_seg = segs_from_subjects(mdd_va)
mdd_te_seg = segs_from_subjects(mdd_te)
hc_tr_seg  = segs_from_subjects(hc_tr)
hc_va_seg  = segs_from_subjects(hc_va)
hc_te_seg  = segs_from_subjects(hc_te)

print("\nMDD subjects train/val/test:", len(mdd_tr), len(mdd_va), len(mdd_te))
print("HC  subjects train/val/test:", len(hc_tr),  len(hc_va),  len(hc_te))
print("MDD segments train/val/test:", len(mdd_tr_seg), len(mdd_va_seg), len(mdd_te_seg))
print("HC  segments train/val/test:", len(hc_tr_seg),  len(hc_va_seg),  len(hc_te_seg))

if len(mdd_tr_seg)==0 or len(hc_tr_seg)==0:
    raise RuntimeError("Training segments empty. Label/subject parsing likely broken.")

# ============================================================
# Model (paper equations; activation optional)
# ============================================================
class EncoderBlock(nn.Module):
    def __init__(self, channels: int, use_act: bool):
        super().__init__()
        self.bn = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=1)
        self.use_act = use_act
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bn(x)
        y = self.conv1(b)
        if self.use_act: y = self.act(y)
        y = y + b
        y = self.conv2(y)
        if self.use_act: y = self.act(y)
        return y

class GDNEncoder(nn.Module):
    def __init__(self, in_channels, latent_dim=300, num_blocks=6, wavelet_len=EXPECTED_WLEN, use_act=False):
        super().__init__()
        self.blocks = nn.ModuleList([EncoderBlock(in_channels, use_act) for _ in range(num_blocks)])
        self.fc_out = nn.Linear(in_channels * wavelet_len, latent_dim)
        self.wavelet_len = wavelet_len

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        b,c,l = x.shape
        x = x.reshape(b, c*l)
        z = self.fc_out(x)
        return z

class DecoderBlock(nn.Module):
    def __init__(self, dim: int, use_act: bool, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.use_act = use_act
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var  = x.var(dim=1, unbiased=False, keepdim=True)
        nrm  = (x - mean) / torch.sqrt(var + self.eps)
        h = self.fc1(nrm)
        if self.use_act: h = self.act(h)
        y = self.fc2(h + nrm)
        if self.use_act: y = self.act(y)
        return y

class GDNDecoder(nn.Module):
    def __init__(self, latent_dim=300, out_len=EXPECTED_WLEN, num_blocks=5, use_act=False):
        super().__init__()
        self.blocks = nn.ModuleList([DecoderBlock(latent_dim, use_act) for _ in range(num_blocks)])
        self.fc_out = nn.Linear(latent_dim, out_len)

    def forward(self, z):
        x = z
        for blk in self.blocks:
            x = blk(x)
        return self.fc_out(x)

class GDNGenerator(nn.Module):
    def __init__(self, in_channels, latent_dim=300, out_len=EXPECTED_WLEN, use_act=False):
        super().__init__()
        self.enc_cA = GDNEncoder(in_channels=in_channels, latent_dim=latent_dim, wavelet_len=out_len, use_act=use_act)
        self.enc_cD = GDNEncoder(in_channels=in_channels, latent_dim=latent_dim, wavelet_len=out_len, use_act=use_act)
        self.w1 = nn.Parameter(torch.tensor(0.5))
        self.w2 = nn.Parameter(torch.tensor(0.5))
        self.dec_cA = GDNDecoder(latent_dim=latent_dim, out_len=out_len, use_act=use_act)
        self.dec_cD = GDNDecoder(latent_dim=latent_dim, out_len=out_len, use_act=use_act)

    def forward(self, ScA, ScD):
        zA = self.enc_cA(ScA)
        zD = self.enc_cD(ScD)
        z = self.w1 * zA + self.w2 * zD
        return self.dec_cA(z), self.dec_cD(z)

# ============================================================
# Dataset
# ============================================================
class GeneratorDataset(Dataset):
    def __init__(self, segments, seg_ids, include_target=False):
        self.segments = segments
        self.seg_ids = list(seg_ids)
        self.include_target = include_target
        self.Lw = segments[self.seg_ids[0]].cA_all.shape[1]
        self.K  = segments[self.seg_ids[0]].sim_idx.shape[1]

    def __len__(self):
        return len(self.seg_ids) * 128

    def __getitem__(self, idx):
        seg_local = idx // 128
        ch = idx % 128
        seg = self.segments[self.seg_ids[seg_local]]

        sim = seg.sim_idx[ch]  # (K,)
        if self.include_target:
            sim = np.concatenate(([ch], sim), axis=0)  # (K+1,)

        ScA = seg.cA_all[sim]      # (C,Lw)
        ScD = seg.cD_all[sim]
        OcA = seg.cA_all[ch]       # (Lw,)
        OcD = seg.cD_all[ch]

        return (torch.from_numpy(ScA).float(),
                torch.from_numpy(ScD).float(),
                torch.from_numpy(OcA).float(),
                torch.from_numpy(OcD).float())

# ============================================================
# Train
# ============================================================
def train_generator(model, loader, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY, name="GDN"):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    for ep in range(1, epochs+1):
        model.train()
        total = 0.0
        nb = 0
        for ScA, ScD, OcA, OcD in loader:
            ScA, ScD = ScA.to(DEVICE), ScD.to(DEVICE)
            OcA, OcD = OcA.to(DEVICE), OcD.to(DEVICE)

            opt.zero_grad(set_to_none=True)
            pcA, pcD = model(ScA, ScD)
            loss = F.mse_loss(pcA, OcA) + F.mse_loss(pcD, OcD)
            loss.backward()
            opt.step()

            total += float(loss.item())
            nb += 1

        if ep == 1 or ep % 5 == 0:
            print(f"[{name}] epoch {ep:03d} | loss {total/max(1,nb):.6f}")
    return model

# ============================================================
# Evaluation (time-domain MSE after iDWT)
# ============================================================
@torch.no_grad()
def reconstruction_mse_time(model, seg: SegmentInfo, include_target=False, batch_size=128):
    model.eval()
    cA_all = seg.cA_all
    cD_all = seg.cD_all
    sim = seg.sim_idx

    if include_target:
        sim2 = np.concatenate([np.arange(128)[:,None], sim], axis=1)  # (128,K+1)
        ScA_np = cA_all[sim2]
        ScD_np = cD_all[sim2]
    else:
        ScA_np = cA_all[sim]
        ScD_np = cD_all[sim]

    ScA = torch.from_numpy(ScA_np).float().to(DEVICE)
    ScD = torch.from_numpy(ScD_np).float().to(DEVICE)

    pred_cA, pred_cD = [], []
    for st in range(0, 128, batch_size):
        ed = min(st+batch_size, 128)
        pcA, pcD = model(ScA[st:ed], ScD[st:ed])
        pred_cA.append(pcA.cpu().numpy())
        pred_cD.append(pcD.cpu().numpy())
    pred_cA = np.concatenate(pred_cA, axis=0)
    pred_cD = np.concatenate(pred_cD, axis=0)

    errs = np.zeros(128, dtype=np.float32)
    for ch in range(128):
        rec_pred = idwt_to_len(pred_cA[ch], pred_cD[ch], out_len=SEG_LEN)
        rec_true = idwt_to_len(cA_all[ch],  cD_all[ch],  out_len=SEG_LEN)
        diff = rec_pred - rec_true
        errs[ch] = float(np.mean(diff**2))
    return errs

def evaluate_segments(gen_mdd, gen_hc, seg_ids, include_target=False):
    out = []
    for sid in seg_ids:
        seg = segments[sid]
        e_mdd = reconstruction_mse_time(gen_mdd, seg, include_target=include_target)
        e_hc  = reconstruction_mse_time(gen_hc,  seg, include_target=include_target)
        n_mdd = int((e_mdd < e_hc).sum())
        out.append({
            "segment_id": sid,
            "subject_id": seg.subject_id,
            "true_label": seg.label,
            "n_mdd_elec": n_mdd
        })
    return out

def find_best_threshold_val(results):
    counts = [r["n_mdd_elec"] for r in results]
    labels = [r["true_label"] for r in results]
    best = (-1.0, None)
    for n0 in range(0, 129):
        preds = [1 if n > n0 else 0 for n in counts]
        acc = sum(int(p==y) for p,y in zip(preds, labels)) / max(1, len(labels))
        if acc > best[0]:
            best = (acc, n0)
    return best[1], best[0]

def metrics(results, n0):
    y = np.array([r["true_label"] for r in results], dtype=int)
    n = np.array([r["n_mdd_elec"] for r in results], dtype=int)
    yp = (n > n0).astype(int)
    tp = int(((yp==1)&(y==1)).sum())
    tn = int(((yp==0)&(y==0)).sum())
    fp = int(((yp==1)&(y==0)).sum())
    fn = int(((yp==0)&(y==1)).sum())
    acc  = (tp+tn)/max(1,len(y))
    sens = tp/max(1,(tp+fn))
    spec = tn/max(1,(tn+fp))
    return {"acc":acc,"sens":sens,"spec":spec,"tp":tp,"tn":tn,"fp":fp,"fn":fn}

def subject_level_acc(results, n0):
    by_subj = defaultdict(list)
    for r in results:
        pred = 1 if r["n_mdd_elec"] > n0 else 0
        by_subj[r["subject_id"]].append((pred, r["true_label"]))
    correct = 0
    for sid, items in by_subj.items():
        preds = [p for p,_ in items]
        true  = items[0][1]
        maj = 1 if sum(preds) >= (len(preds)/2.0) else 0
        correct += int(maj == true)
    return correct / max(1, len(by_subj))

# ============================================================
# Run
# ============================================================
train_include = INCLUDE_TARGET_IN_ENCODER_INPUT
in_channels = (K_SIM + 1) if train_include else K_SIM

mdd_train_ds = GeneratorDataset(segments, mdd_tr_seg, include_target=train_include)
hc_train_ds  = GeneratorDataset(segments, hc_tr_seg,  include_target=train_include)

mdd_loader = DataLoader(mdd_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
hc_loader  = DataLoader(hc_train_ds,  batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

gen_mdd = GDNGenerator(in_channels=in_channels, latent_dim=300, out_len=mdd_train_ds.Lw, use_act=USE_ACTIVATION)
gen_hc  = GDNGenerator(in_channels=in_channels, latent_dim=300, out_len=mdd_train_ds.Lw, use_act=USE_ACTIVATION)

print("\n=== Train MDD Generator ===")
gen_mdd = train_generator(gen_mdd, mdd_loader, name="MDD")

print("\n=== Train HC Generator ===")
gen_hc  = train_generator(gen_hc,  hc_loader,  name="HC")

print("\n=== Validation: choose n0 ===")
val_ids = mdd_va_seg + hc_va_seg
val_res = evaluate_segments(gen_mdd, gen_hc, val_ids, include_target=train_include)
n0, val_acc = find_best_threshold_val(val_res)
print(f"Chosen n0={n0} | val acc={val_acc*100:.2f}%")

print("\n=== Test ===")
test_ids = mdd_te_seg + hc_te_seg
test_res = evaluate_segments(gen_mdd, gen_hc, test_ids, include_target=train_include)
m = metrics(test_res, n0)
print(f"Segment acc={m['acc']*100:.2f}% | sens={m['sens']*100:.2f}% | spec={m['spec']*100:.2f}%")
print("Confusion:", {k:m[k] for k in ["tp","tn","fp","fn"]})

sa = subject_level_acc(test_res, n0)
print(f"Subject-level acc={sa*100:.2f}%")

print("\n[Sanity] Key settings:")
print("  LABEL_MODE =", LABEL_MODE)
print("  DEMEAN_PER_CHANNEL =", DEMEAN_PER_CHANNEL)
print("  ZSCORE_PER_CHANNEL =", ZSCORE_PER_CHANNEL)
print("  SIMILARITY_ON_FILTERED =", SIMILARITY_ON_FILTERED)
print("  INCLUDE_TARGET_IN_ENCODER_INPUT =", INCLUDE_TARGET_IN_ENCODER_INPUT)
print("  HAMMING_MODE =", HAMMING_MODE)
print("  DWT_MODE =", DWT_MODE)
print("  USE_ACTIVATION =", USE_ACTIVATION)
print("  EXPECTED_WLEN =", EXPECTED_WLEN)

Mounted at /content/drive
Torch: 2.9.0+cu126 | Device: cuda
Unzipping (overwrite, quiet)...
Found .mat: 53
Unique subject IDs: 53

[Label:auto_prefix] selected prefix_len = 4
[Label:auto_prefix] prefix groups (prefix -> #subjects):
  ### 
  ### 
  ### 
[Label:auto_prefix] implied counts: MDD=24, HC=29, target(MDD,HC)=(24,29), score=0
[Label] Subjects: total=53 | label1=24 | label0=29

Precomputing: similarity -> FFT(4-14) -> db6 DWT ...


100%|██████████| 53/53 [00:09<00:00,  5.49it/s]


Segments prepared: 424 | EXPECTED_WLEN=1255

MDD subjects train/val/test: 15 5 4
HC  subjects train/val/test: 15 5 9
MDD segments train/val/test: 120 40 32
HC  segments train/val/test: 120 40 72

=== Train MDD Generator ===
[MDD] epoch 001 | loss 361.821334
[MDD] epoch 005 | loss 215.008472
[MDD] epoch 010 | loss 223.319250
[MDD] epoch 015 | loss 214.398806
[MDD] epoch 020 | loss 236.450931
[MDD] epoch 025 | loss 200.461353
[MDD] epoch 030 | loss 200.975203
[MDD] epoch 035 | loss 215.553199
[MDD] epoch 040 | loss 197.174603
[MDD] epoch 045 | loss 205.097396
[MDD] epoch 050 | loss 194.332595
[MDD] epoch 055 | loss 196.611063
[MDD] epoch 060 | loss 195.591373
[MDD] epoch 065 | loss 225.600728
[MDD] epoch 070 | loss 201.196913
[MDD] epoch 075 | loss 189.832998
[MDD] epoch 080 | loss 191.671671
[MDD] epoch 085 | loss 190.151388
[MDD] epoch 090 | loss 193.987567
[MDD] epoch 095 | loss 185.121407
[MDD] epoch 100 | loss 190.275043
[MDD] epoch 105 | loss 182.008705
[MDD] epoch 110 | loss 189.9

# GICN Switch-Sensitivity Mini-Study

In [ ]:
# ================================
# EEG-SOLO mini proof: ambiguity -> performance moves
# Single Colab cell, based on the attached re_impl.ipynb (GICN / Improved GCN pipeline).
#
# What this runs:
#   - One representative GCN pipeline (GICN / "Improved GCN") on MODMA
#   - 3 binary switches -> 8 runs
#       (1) NORM_FIT_SCOPE: "train" (leak-safe) vs "global" (leaky/optimistic demo)
#       (2) ADJ_PER_SEGMENT: per-window adjacency vs per-subject adjacency
#       (3) SUBJECT_AGG_RULE: subject aggregation = mean(prob) vs majority-vote
#   - Reports subject-level accuracy (and AUC/recall/spec/precision) for each config.
#
# Notes:
#   - Defaults are set for a "mini" demo (fewer epochs + optional window subsampling).
#   - For paper-quality numbers, set:
#         FAST_MINI=False  (or raise NUM_EPOCHS, set WINDOW_STRIDE=1, MAX_WINDOWS=None, N_SPLITS=10)
# ================================

# (Optional) dependencies. In Colab most are available; uncomment if needed.
# !pip -q install numpy scipy scikit-learn tqdm pandas

import os, re, glob, zipfile, random, math, copy, pickle, gzip
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from scipy.io import loadmat
from scipy.signal import welch, firwin, filtfilt
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, confusion_matrix
import pandas as pd

# display() in Colab/Jupyter
try:
    from IPython.display import display  # type: ignore
except Exception:
    def display(x):
        print(x)

# ---- Colab Drive (optional) ----
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except Exception as e:
    print("[info] google.colab not available (ok outside Colab):", repr(e))

# -----------------------
# User settings (edit me)
# -----------------------
ZIP_PATH  = "/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015.zip"
DATA_ROOT = "/content/modma_lanzhou"  # extracted folder root (will be created if missing)
LABEL_CSV = None  # e.g., "/content/modma_lanzhou/labels.csv" (recommended if you have a definitive mapping)

CACHE_DIR = "/content/gicn_cache"  # cached subject features/graphs to avoid recomputation
os.makedirs(CACHE_DIR, exist_ok=True)

# -----------------------
# Mini vs full settings
# -----------------------
FAST_MINI = True

# Segmentation constants (paper-aligned in the attached code)
FS = 250
WIN_SEC  = 4.0
STEP_SEC = 0.5
MAX_SEC  = 200.0  # matches 393 windows with (4s, 0.5s shift) over 200 seconds

# Optional speed knobs for the mini demo (set to paper-quality values for final runs)
WINDOW_STRIDE = 3 if FAST_MINI else 1       # keep every k-th window (mini demo)
MAX_WINDOWS   = 150 if FAST_MINI else None  # cap windows per subject (mini demo)

# CV / training hyperparams
N_SPLITS   = 5 if FAST_MINI else 10
SEED       = 42
NUM_EPOCHS = 15 if FAST_MINI else 60
BATCH_SIZE = 64
BASE_LR    = 0.005
LR_STEP    = 30
LR_GAMMA   = 0.1
DROPOUT    = 0.2
DENSE_DIM  = 6

# Filtering (kept as in the attached implementation)
BANDPASS_LO = 0.5
BANDPASS_HI = 45.0
FIR_TAPS    = 101

# Subject grouping & label heuristics (same spirit as the attached code)
GROUP_BY_SUBJECT_ID = True


# -----------------------
# Utilities
# -----------------------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

def ensure_unzipped(zip_path: str, dst_root: str):
    os.makedirs(dst_root, exist_ok=True)
    mat_glob = glob.glob(os.path.join(dst_root, "**", "*.mat"), recursive=True)
    if mat_glob:
        print(f"[data] Found {len(mat_glob)} .mat files under {dst_root} (skip unzip).")
        return
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Zip not found: {zip_path}\nFix ZIP_PATH.")
    print("[data] Extracting zip... (first run only)")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dst_root)
    mat_glob = glob.glob(os.path.join(dst_root, "**", "*.mat"), recursive=True)
    print(f"[data] Now found {len(mat_glob)} .mat files under {dst_root}.")

def load_label_map(label_csv_path: str):
    # Minimal CSV reader: "subject_id,label" per row (no header assumed).
    mp = {}
    with open(label_csv_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 2:
                continue
            sid, y = parts[0], parts[1]
            if sid.lower() in ("subject", "subject_id"):
                continue
            mp[str(sid)] = int(y)
    return mp

def parse_subject_id(path: str) -> str:
    base = os.path.basename(path)
    m = re.search(r"(\d{4,})", base)
    if m:
        return m.group(1)
    return base

def parse_label(path: str, subject_id: str, label_map=None) -> int:
    if label_map is not None and subject_id in label_map:
        return int(label_map[subject_id])
    # Heuristic fallback (NOT recommended; matches the attached code's spirit)
    low = path.lower()
    if "mdd" in low or "depress" in low:
        return 1
    if "hc" in low or "control" in low or "healthy" in low:
        return 0
    # MODMA-ish heuristic used in the attached code
    if subject_id.startswith("0201"):
        return 1
    return 0

def group_mat_files_by_subject(mat_files):
    groups = {}
    for p in mat_files:
        sid = parse_subject_id(p)
        groups.setdefault(sid, []).append(p)
    # deterministic order
    for sid in groups:
        groups[sid] = sorted(groups[sid])
    return groups

def load_eeg_mat(path: str):
    """
    Robust loader: finds a 2D array whose one dimension is 128 or 129.
    Returns shape (128, T) float32.
    """
    mat = loadmat(path)
    candidates = []
    for k, v in mat.items():
        if k.startswith("__"):
            continue
        if isinstance(v, np.ndarray) and v.ndim == 2:
            candidates.append((k, v))
    if not candidates:
        raise ValueError(f"No 2D array found in {path}")
    # pick first plausible
    x = None
    for k, v in candidates:
        if 128 in v.shape or 129 in v.shape:
            x = v
            break
    if x is None:
        x = candidates[0][1]
    # ensure shape (C, T)
    if x.shape[0] not in (128, 129) and x.shape[1] in (128, 129):
        x = x.T
    if x.shape[0] == 129:
        x = x[:128, :]
    if x.shape[0] != 128:
        raise ValueError(f"EEG array has shape {x.shape} (expected 128/129 channels)")
    return x.astype(np.float32)

def bandpass_fir(x, fs=FS, lo=BANDPASS_LO, hi=BANDPASS_HI, taps=FIR_TAPS):
    nyq = fs / 2.0
    b = firwin(taps, [lo/nyq, hi/nyq], pass_zero=False)
    return filtfilt(b, [1.0], x, axis=1).astype(np.float32)

def segment_signal(data, fs=FS, win_sec=WIN_SEC, step_sec=STEP_SEC, max_sec=MAX_SEC):
    win = int(win_sec * fs)
    step = int(step_sec * fs)
    max_samp = int(max_sec * fs)
    x = data[:, :min(data.shape[1], max_samp)]
    T = x.shape[1]
    if T < win:
        return []
    segs = []
    for start in range(0, T - win + 1, step):
        segs.append(x[:, start:start+win])
    # speed knobs
    if WINDOW_STRIDE and WINDOW_STRIDE > 1:
        segs = segs[::WINDOW_STRIDE]
    if MAX_WINDOWS is not None:
        segs = segs[:MAX_WINDOWS]
    return segs

# -----------------------
# Feature extraction
# -----------------------
def hjorth_params(x):
    # x: (C, T)
    dx = np.diff(x, axis=1)
    ddx = np.diff(dx, axis=1)
    var0 = np.var(x, axis=1) + 1e-8
    var1 = np.var(dx, axis=1) + 1e-8
    var2 = np.var(ddx, axis=1) + 1e-8
    activity = var0
    mobility = np.sqrt(var1 / var0)
    complexity = np.sqrt(var2 / var1) / mobility
    return activity, mobility, complexity

def psd_bandpower(x, fs=FS):
    # x: (C, T)
    # Using Welch; take total power in 0.5-45Hz as one scalar feature per channel
    f, Pxx = welch(x, fs=fs, nperseg=min(256, x.shape[1]), axis=1)
    mask = (f >= BANDPASS_LO) & (f <= BANDPASS_HI)
    power = np.trapz(Pxx[:, mask], f[mask], axis=1)
    return power

def compute_node_features(seg):
    # seg: (128, win)
    act, mob, comp = hjorth_params(seg)
    pwr = psd_bandpower(seg)
    feat = np.stack([act, mob, comp, pwr], axis=1).astype(np.float32)  # (128, 4)
    return feat

def normalized_abs_corr(seg):
    # seg: (128, win)
    c = np.corrcoef(seg)
    c = np.nan_to_num(c, nan=0.0, posinf=0.0, neginf=0.0)
    A = np.abs(c).astype(np.float32)
    np.fill_diagonal(A, 1.0)
    d = np.sum(A, axis=1) + 1e-8
    Dinv = np.diag(1.0 / np.sqrt(d))
    An = Dinv @ A @ Dinv
    return An.astype(np.float32)

# -----------------------
# Dataset cache builder
# -----------------------
def build_subject_cache(mat_files, label_map=None, adj_per_segment=True):
    if GROUP_BY_SUBJECT_ID:
        groups = group_mat_files_by_subject(mat_files)
        subject_ids = sorted(groups.keys())
        file_lists = [groups[sid] for sid in subject_ids]
    else:
        subject_ids = [parse_subject_id(p) for p in mat_files]
        file_lists = [[p] for p in mat_files]

    labels = []
    subj_X, subj_A = [], []
    kept_ids = []

    if label_map is None:
        print("WARNING: LABEL_CSV is not provided. Labels will be inferred from file names / heuristics.")
        print("         For a reproducibility study, providing LABEL_CSV is strongly recommended.")

    for sid, paths in tqdm(list(zip(subject_ids, file_lists)), desc="Loading subjects"):
        y = parse_label(paths[0], sid, label_map=label_map)

        # deterministic file choice: first path (aligned with "paper-only deterministic rule" spirit)
        eeg = load_eeg_mat(paths[0])  # (128, T)
        eeg = bandpass_fir(eeg)

        segments = segment_signal(eeg)
        if len(segments) == 0:
            print(f"Warning: no segments for {sid}, skip")
            continue

        X = np.stack([compute_node_features(seg) for seg in segments], axis=0).astype(np.float32)  # (n_seg,128,4)

        if adj_per_segment:
            A = np.stack([normalized_abs_corr(seg) for seg in segments], axis=0).astype(np.float16)
        else:
            A0 = normalized_abs_corr(eeg[:, :min(eeg.shape[1], int(MAX_SEC*FS))])
            A  = np.repeat(A0[None, :, :], repeats=X.shape[0], axis=0).astype(np.float16)

        kept_ids.append(sid)
        subj_X.append(X)
        subj_A.append(A)
        labels.append(y)

    subj_y = np.array(labels, dtype=np.int64)
    return kept_ids, subj_y, subj_X, subj_A

def cache_path(adj_per_segment: bool) -> str:
    tag = f"adjperseg{int(adj_per_segment)}_maxsec{int(MAX_SEC)}_stride{int(WINDOW_STRIDE)}_cap{MAX_WINDOWS if MAX_WINDOWS is not None else 'none'}"
    return os.path.join(CACHE_DIR, f"gicn_cache_{tag}.pkl.gz")

def save_cache(obj, path: str):
    with gzip.open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_cache(path: str):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

def get_subject_cache(mat_files, label_map, adj_per_segment: bool):
    path = cache_path(adj_per_segment)
    if os.path.exists(path):
        print(f"[cache] Loading: {path}")
        return load_cache(path)
    print(f"[cache] Building cache (adj_per_segment={adj_per_segment}) ...")
    subject_ids, subj_y, subj_X, subj_A = build_subject_cache(mat_files, label_map=label_map, adj_per_segment=adj_per_segment)
    obj = {"subject_ids": subject_ids, "subj_y": subj_y, "subj_X": subj_X, "subj_A": subj_A}
    save_cache(obj, path)
    print(f"[cache] Saved: {path}")
    return obj

# -----------------------
# Torch dataset
# -----------------------
class GraphSegmentDataset(Dataset):
    def __init__(self, subject_indices, subj_X, subj_A, subj_y, return_sub_idx=False, feat_mean=None, feat_std=None):
        self.subj_X = subj_X
        self.subj_A = subj_A
        self.subj_y = subj_y
        self.return_sub_idx = return_sub_idx
        self.feat_mean = feat_mean
        self.feat_std = feat_std
        self.samples = []
        for s in subject_indices:
            n_seg = subj_X[s].shape[0]
            for k in range(n_seg):
                self.samples.append((int(s), int(k)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s, k = self.samples[idx]
        x = self.subj_X[s][k]                    # (N,4)
        if self.feat_mean is not None and self.feat_std is not None:
            x = (x - self.feat_mean) / self.feat_std
        a = self.subj_A[s][k].astype(np.float32) # (N,N)
        y = float(self.subj_y[s])
        if self.return_sub_idx:
            return torch.from_numpy(x), torch.from_numpy(a), torch.tensor(y, dtype=torch.float32), torch.tensor(s, dtype=torch.long)
        return torch.from_numpy(x), torch.from_numpy(a), torch.tensor(y, dtype=torch.float32)

def compute_feature_stats(subject_indices, subj_X):
    """Compute feature-wise mean/std from the provided subjects."""
    xs = []
    for s in subject_indices:
        x = subj_X[int(s)]  # (n_seg, N, 4)
        xs.append(x.reshape(-1, x.shape[-1]))
    arr = np.concatenate(xs, axis=0)
    mean = arr.mean(axis=0).astype(np.float32)
    std = (arr.std(axis=0) + 1e-8).astype(np.float32)
    return mean.reshape(1, -1), std.reshape(1, -1)

# -----------------------
# Model (GICN / Improved GCN)
# -----------------------
class GraphConvLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.w = nn.Parameter(torch.randn(in_ch, out_ch) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_ch))

    def forward(self, x, adj_norm):
        # x: (B, N, in_ch), adj_norm: (B, N, N)
        h = torch.matmul(x, self.w) + self.b
        return torch.matmul(adj_norm, h)

class GraphInputLayer(nn.Module):
    """Learnable weighting on nodes (paper's 'improved input layer' spirit)."""
    def __init__(self, num_nodes, in_ch, out_ch):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(num_nodes, 1))  # (N,1)
        self.w = nn.Parameter(torch.randn(in_ch, out_ch) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_ch))

    def forward(self, x, adj_norm):
        # x: (B,N,in_ch)
        xw = x * self.alpha.unsqueeze(0)  # (B,N,in_ch)
        h = torch.matmul(xw, self.w) + self.b
        return torch.matmul(adj_norm, h)

class GICN(nn.Module):
    def __init__(self, num_nodes=128, in_channels=4, dense_dim=DENSE_DIM, dropout=DROPOUT):
        super().__init__()
        self.input_layer = GraphInputLayer(num_nodes, in_channels, 4)
        self.bn1 = nn.BatchNorm1d(4)

        self.gcn2 = GraphConvLayer(4, 8)
        self.bn2 = nn.BatchNorm1d(8)

        self.gcn3 = GraphConvLayer(8, 16)
        self.bn3 = nn.BatchNorm1d(16)

        self.act = nn.LeakyReLU(0.2)
        self.fc1 = nn.Linear(num_nodes * 16, dense_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(dense_dim, 1)

    def forward(self, x, adj_norm):
        h = self.act(self.input_layer(x, adj_norm))
        h = self.bn1(h.permute(0,2,1)).permute(0,2,1)

        h = self.act(self.gcn2(h, adj_norm))
        h = self.bn2(h.permute(0,2,1)).permute(0,2,1)

        h = self.act(self.gcn3(h, adj_norm))
        h = self.bn3(h.permute(0,2,1)).permute(0,2,1)

        h = h.reshape(h.size(0), -1)
        h = self.dropout(self.act(self.fc1(h)))
        out = self.fc_out(h)  # logits
        return out.squeeze(-1)

# -----------------------
# Evaluation (subject-level aggregation switch)
# -----------------------
@torch.no_grad()
def eval_subject_level(model, loader, device, agg_rule="meanprob"):
    model.eval()
    all_logits, all_y, all_sub = [], [], []
    for batch in loader:
        xb, ab, yb, sb = batch
        xb = xb.to(device)
        ab = ab.to(device)
        yb = yb.to(device)
        logits = model(xb, ab)
        all_logits.append(logits.detach().cpu())
        all_y.append(yb.detach().cpu())
        all_sub.append(sb.detach().cpu())
    logits = torch.cat(all_logits).numpy()
    y = torch.cat(all_y).numpy().astype(np.int64)
    subs = torch.cat(all_sub).numpy().astype(np.int64)
    probs = 1.0 / (1.0 + np.exp(-logits))

    subj_probs = {}
    subj_ys = {}
    subj_votes = {}
    for p, yy, s in zip(probs, y, subs):
        s = int(s)
        subj_probs.setdefault(s, []).append(float(p))
        subj_votes.setdefault(s, []).append(int(p >= 0.5))
        subj_ys[s] = int(yy)

    agg_probs = []
    agg_pred = []
    agg_y = []
    for s in sorted(subj_probs.keys()):
        ps = subj_probs[s]
        vs = subj_votes[s]
        if agg_rule == "majorityvote":
            pred = 1 if (sum(vs) >= (len(vs) / 2.0)) else 0
            prob = float(np.mean(ps))  # still used for AUC
        else:
            prob = float(np.mean(ps))
            pred = 1 if prob >= 0.5 else 0
        agg_probs.append(prob)
        agg_pred.append(pred)
        agg_y.append(subj_ys[s])

    agg_probs = np.array(agg_probs)
    agg_pred = np.array(agg_pred)
    agg_y = np.array(agg_y)

    acc = accuracy_score(agg_y, agg_pred)
    rec = recall_score(agg_y, agg_pred, zero_division=0)
    prec = precision_score(agg_y, agg_pred, zero_division=0)
    try:
        auc = roc_auc_score(agg_y, agg_probs)
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(agg_y, agg_pred, labels=[0,1]).ravel()
    spec = tn / (tn + fp + 1e-8)
    return {"acc": acc, "auc": auc, "recall": rec, "spec": spec, "prec": prec}

# -----------------------
# Train / CV
# -----------------------
def train_one_fold(train_sub_idx, test_sub_idx, subj_X, subj_A, subj_y, device,
                   norm_fit_scope="train", agg_rule="meanprob"):
    # nested val split (subject-wise) for early stopping model selection
    if len(train_sub_idx) >= 6:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
        tr_idx, va_idx = next(sss.split(train_sub_idx, subj_y[train_sub_idx]))
        tr_sub = train_sub_idx[tr_idx]
        va_sub = train_sub_idx[va_idx]
    else:
        tr_sub = train_sub_idx
        va_sub = None

    # --- normalization stats ---
    if norm_fit_scope == "global":
        # INTENTIONALLY LEAKY DEMO: fit on all subjects (train+test)
        all_idx = np.arange(len(subj_X))
        feat_mean, feat_std = compute_feature_stats(all_idx, subj_X)
    else:
        # leak-safe: fit on training subjects only
        feat_mean, feat_std = compute_feature_stats(tr_sub, subj_X)

    train_ds = GraphSegmentDataset(tr_sub, subj_X, subj_A, subj_y, return_sub_idx=False, feat_mean=feat_mean, feat_std=feat_std)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)

    if va_sub is not None:
        val_ds = GraphSegmentDataset(va_sub, subj_X, subj_A, subj_y, return_sub_idx=False, feat_mean=feat_mean, feat_std=feat_std)
        val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0)
    else:
        val_ld = None

    # test loader must return subject ids for subject-level aggregation
    test_ds = GraphSegmentDataset(test_sub_idx, subj_X, subj_A, subj_y, return_sub_idx=True, feat_mean=feat_mean, feat_std=feat_std)
    test_ld = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0)

    num_nodes = subj_X[0].shape[1]
    model = GICN(num_nodes=num_nodes, in_channels=4, dense_dim=DENSE_DIM, dropout=DROPOUT).to(device)
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=BASE_LR)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_STEP, gamma=LR_GAMMA)

    best_state = None
    best_val = -1.0

    for ep in range(1, NUM_EPOCHS + 1):
        model.train()
        run_loss = 0.0
        for batch in train_ld:
            xb, ab, yb = batch
            xb = xb.to(device)
            ab = ab.to(device)
            yb = yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb, ab)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            run_loss += loss.item() * xb.size(0)

        sch.step()

        if val_ld is not None:
            # simple val acc (segment-level), just for model selection sanity
            model.eval()
            ys, ps = [], []
            with torch.no_grad():
                for xb, ab, yb in val_ld:
                    xb, ab = xb.to(device), ab.to(device)
                    logits = model(xb, ab).detach().cpu().numpy()
                    probs = 1.0 / (1.0 + np.exp(-logits))
                    ys.append(yb.numpy())
                    ps.append(probs)
            ys = np.concatenate(ys).astype(np.int64)
            ps = np.concatenate(ps)
            pred = (ps >= 0.5).astype(np.int64)
            val_acc = accuracy_score(ys, pred)
            if val_acc > best_val:
                best_val = val_acc
                best_state = copy.deepcopy(model.state_dict())
        else:
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    m_test = eval_subject_level(model, test_ld, device, agg_rule=agg_rule)
    return m_test

def summarize_metrics(fold_metrics, key, as_percent=True):
    vals = [m[key] for m in fold_metrics if not (isinstance(m[key], float) and np.isnan(m[key]))]
    if not vals:
        return float("nan"), float("nan")
    mean = float(np.mean(vals))
    std = float(np.std(vals))
    if as_percent:
        return 100.0*mean, 100.0*std
    return mean, std

def run_cv(subj_y, subj_X, subj_A, norm_fit_scope="train", agg_rule="meanprob"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    indices = np.arange(len(subj_y))
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_metrics = []
    for fold, (tr, te) in enumerate(skf.split(indices, subj_y), 1):
        # reset RNG so differences are attributable to switch settings, not random init/order
        set_seed(SEED + fold)
        m = train_one_fold(tr, te, subj_X, subj_A, subj_y, device,
                           norm_fit_scope=norm_fit_scope, agg_rule=agg_rule)
        fold_metrics.append(m)
    mean_acc, std_acc = summarize_metrics(fold_metrics, "acc", as_percent=True)
    mean_auc, std_auc = summarize_metrics(fold_metrics, "auc", as_percent=False)
    mean_rec, std_rec = summarize_metrics(fold_metrics, "recall", as_percent=True)
    mean_spec, std_spec = summarize_metrics(fold_metrics, "spec", as_percent=True)
    mean_prec, std_prec = summarize_metrics(fold_metrics, "prec", as_percent=True)
    return {
        "acc_mean": mean_acc, "acc_std": std_acc,
        "auc_mean": mean_auc, "auc_std": std_auc,
        "recall_mean": mean_rec, "recall_std": std_rec,
        "spec_mean": mean_spec, "spec_std": std_spec,
        "prec_mean": mean_prec, "prec_std": std_prec,
    }

# -----------------------
# 8-run switch experiment
# -----------------------
ensure_unzipped(ZIP_PATH, DATA_ROOT)
mat_files = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.mat"), recursive=True))
print("[data] mat files:", len(mat_files))

label_map = load_label_map(LABEL_CSV) if LABEL_CSV else None

# Build/load the 2 caches (adj_per_segment True/False) once
cache_seg = get_subject_cache(mat_files, label_map, adj_per_segment=True)
cache_sub = get_subject_cache(mat_files, label_map, adj_per_segment=False)

def run_one_config(adj_per_segment: bool, norm_fit_scope: str, agg_rule: str):
    cache = cache_seg if adj_per_segment else cache_sub
    subj_y = cache["subj_y"]
    subj_X = cache["subj_X"]
    subj_A = cache["subj_A"]
    return run_cv(subj_y, subj_X, subj_A, norm_fit_scope=norm_fit_scope, agg_rule=agg_rule)

configs = []
for adj_per_segment in [True, False]:
    for norm_fit_scope in ["train", "global"]:
        for agg_rule in ["meanprob", "majorityvote"]:
            configs.append({
                "ADJ_PER_SEGMENT": adj_per_segment,
                "NORM_FIT_SCOPE": norm_fit_scope,
                "SUBJECT_AGG_RULE": agg_rule,
            })

rows = []
for i, cfg in enumerate(configs, 1):
    print(f"\n=== Run {i}/{len(configs)} === {cfg}")
    out = run_one_config(cfg["ADJ_PER_SEGMENT"], cfg["NORM_FIT_SCOPE"], cfg["SUBJECT_AGG_RULE"])
    rows.append({**cfg, **out})
    print(f"  subject-acc: {out['acc_mean']:.2f}% ± {out['acc_std']:.2f}%  | AUC {out['auc_mean']:.4f} ± {out['auc_std']:.4f}")

df = pd.DataFrame(rows)
df = df.sort_values(["acc_mean"], ascending=False).reset_index(drop=True)

print("\n==================== Results (sorted by subject-acc) ====================")
display(df)

acc_min = float(df["acc_mean"].min())
acc_max = float(df["acc_mean"].max())
print(f"\n[range] subject-level accuracy moved from {acc_min:.2f}% to {acc_max:.2f}% (Δ={acc_max-acc_min:.2f} points)")

# Save CSV for manuscript supplement
out_csv = os.path.join(CACHE_DIR, "gicn_switch_sensitivity_results.csv")
df.to_csv(out_csv, index=False)
print("[saved]", out_csv)

Mounted at /content/drive
[data] Extracting zip... (first run only)
[data] Now found 53 .mat files under /content/modma_lanzhou.
[data] mat files: 53
[cache] Building cache (adj_per_segment=True) ...
         For a reproducibility study, providing LABEL_CSV is strongly recommended.


Loading subjects:   0%|          | 0/53 [00:00<?, ?it/s]

/tmp/ipython-input-1364119716.py:243: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  power = np.trapz(Pxx[:, mask], f[mask], axis=1)


[cache] Saved: /content/gicn_cache/gicn_cache_adjperseg1_maxsec200_stride3_cap150.pkl.gz
[cache] Building cache (adj_per_segment=False) ...
         For a reproducibility study, providing LABEL_CSV is strongly recommended.


Loading subjects:   0%|          | 0/53 [00:00<?, ?it/s]

...

  subject-acc: 51.09% ± 9.87%  | AUC 0.4513 ± 0.1515

=== Run 6/8 === {'ADJ_PER_SEGMENT': False, 'NORM_FIT_SCOPE': 'train', 'SUBJECT_AGG_RULE': 'majorityvote'}


/tmp/ipython-input-1364119716.py:570: RuntimeWarning: overflow encountered in exp
  probs = 1.0 / (1.0 + np.exp(-logits))


  subject-acc: 51.09% ± 9.87%  | AUC 0.4513 ± 0.1515

=== Run 7/8 === {'ADJ_PER_SEGMENT': False, 'NORM_FIT_SCOPE': 'global', 'SUBJECT_AGG_RULE': 'meanprob'}
  subject-acc: 41.45% ± 8.86%  | AUC 0.4440 ± 0.1067

=== Run 8/8 === {'ADJ_PER_SEGMENT': False, 'NORM_FIT_SCOPE': 'global', 'SUBJECT_AGG_RULE': 'majorityvote'}
  subject-acc: 47.09% ± 9.49%  | AUC 0.4440 ± 0.1067

==================== Results (sorted by subject-acc) ====================


,ADJ_PER_SEGMENT,NORM_FIT_SCOPE,SUBJECT_AGG_RULE,acc_mean,acc_std,auc_mean,auc_std,recall_mean,recall_std,spec_mean,spec_std,prec_mean,prec_std
0,True,train,majorityvote,62.363636,7.662294,0.652667,0.105122,55.0,18.439089,69.333333,5.333333,58.666667,7.483315
1,True,train,meanprob,60.545455,10.688683,0.652667,0.105122,47.0,18.867962,72.666667,7.423686,57.000000,9.797959
2,True,global,majorityvote,60.363636,6.645697,0.615333,0.136815,50.0,26.832816,70.000000,22.110832,63.333333,19.436506
3,True,global,meanprob,58.545455,8.862587,0.615333,0.136815,46.0,32.000000,70.000000,22.110832,53.333333,32.317866
4,False,train,meanprob,51.090909,9.865207,0.451333,0.151461,59.0,30.397368,44.666667,12.578641,40.000000,20.995626
5,False,train,majorityvote,51.090909,9.865207,0.451333,0.151461,59.0,30.397368,44.666667,12.578641,40.000000,20.995626
6,False,global,majorityvote,47.090909,9.489446,0.444000,0.106717,47.0,31.559468,50.000000,27.888667,32.500000,19.245991
7,False,global,meanprob,41.454545,8.862587,0.444000,0.106717,43.0,35.721142,42.666667,21.437247,26.071429,21.653581



[range] subject-level accuracy moved from 41.45% to 62.36% (Δ=20.91 points)
[saved] /content/gicn_cache/gicn_switch_sensitivity_results.csv


# SSPA‑GCN Switch‑Sensitivity Mini‑Study

In [ ]:
# ============================
# SSPA‑GCN Switch‑Sensitivity Mini‑Study (MODMA EEG, LOSO) — STABLE FIX
# 3 switches × 2 levels (2^3 = 8):
#   1) SSP_MODE  : "paper" vs "strict"
#   2) ZERO_DIAG : True vs False
#   3) POOLING   : "mean" vs "flatten"
#
# Fix for flatten-collapse:
#   - Use shared bottleneck for flatten pooling: (C*Fout) -> Fout
#   - Apply Eq.(11) regularization only to backbone weights (atten + gcn thetas)
#     to avoid confounding pooling choice with massive L1 penalty.
# ============================

import os, glob, math, random, itertools, subprocess
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.io import loadmat
from scipy.signal import butter, filtfilt


# ----------------------
# User knobs (edit here)
# ----------------------
ZIP_PATH = "/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015.zip"
RAW_ROOT = "/content/modma_raw"

RUN_SWEEP = True          # True: run all 8 configs; False: only reference ("paper", True, "mean")

# mini speed knobs (paper-aligned: TRAIN/EVAL always 150 segments)
FAST_MINI = True
MINI_EPOCHS = 20          # raise to 30-50 if you want less noise
MINI_BATCH  = 300         # paper-like
SSP_MMD_SEGMENTS = 30     # only for SSP/MMD distances; set 150 for full SSP fidelity

LIMIT_N_TEST_SUBJECTS = 0 # 0: all 53 folds; set e.g. 10 for smoke test
VERBOSE_FOLDS = False     # print per-fold predictions (very verbose)


# ======================
# Config
# ======================
@dataclass
class CFG:
    FS: int = 250
    SEG_SECONDS: int = 2
    N_SEGMENTS: int = 150
    N_CHANNELS: int = 128
    BANDS: Tuple[Tuple[float, float], ...] = ((0.5, 4), (4, 8), (8, 12), (12, 35), (35, 100))

    PHQ_MAX: float = 27.0
    PHI: float = 0.3
    LR: float = 0.001
    EPOCHS: int = 50
    BATCH_SIZE: int = 300
    ALPHA_L1: float = 0.002
    BETA_L2: float = 0.0125
    DROPOUT: float = 0.5
    DN: int = 12

    CHEB_K: int = 3
    GCN_HIDDEN: int = 64
    GCN_OUT: int = 128

    ZSCORE_DE_FOR_MODEL: bool = False
    ZSCORE_DE_FOR_SSP: bool = False

    MMD_BANDWIDTH: str = "median"
    MMD_ESTIMATOR: str = "biased"
    SIGMA_MEDIAN_MAX_SAMPLES: int = 1200
    WEIGHTED_MERGE: bool = False

    GRL_MODE: str = "dann"   # "constant" or "dann"
    GRL_LAMBDA: float = 1.0

    DOMAIN_LOSS_WEIGHT: float = 1.0
    CLIP_GRAD_NORM: float = 0.0

    SEED: int = 0
    CROP_MODE: str = "first" # "first" or "random"

cfg = CFG()
if FAST_MINI:
    cfg.EPOCHS = int(MINI_EPOCHS)
    cfg.BATCH_SIZE = int(MINI_BATCH)
    cfg.SIGMA_MEDIAN_MAX_SAMPLES = 400


# ======================
# Reproducibility
# ======================
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] device:", device)


# ======================
# Colab drive mount + unzip
# ======================
def mount_drive_if_colab():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")   # type: ignore
        return True
    except Exception:
        return False

def unzip_if_needed(zip_path: str, dst_root: str):
    os.makedirs(dst_root, exist_ok=True)
    has_mat = any(p.endswith(".mat") for p in glob.glob(os.path.join(dst_root, "**", "*.mat"), recursive=True))
    if has_mat:
        print("[INFO] Found .mat files under", dst_root, "-> skip unzip")
        return
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"ZIP not found: {zip_path}")
    print("[INFO] Unzipping MODMA zip (only once)...")
    subprocess.run(["unzip", "-o", "-qq", zip_path, "-d", dst_root], check=True)
    print("[INFO] Unzip done.")

_ = mount_drive_if_colab()
unzip_if_needed(ZIP_PATH, RAW_ROOT)


# ======================
# PHQ-9 map
# ======================
raise RuntimeError(
    "PHQ_SCORES (subject_id -> PHQ-9) was present in the internal run, but is omitted "
    "in this public release to comply with the MODMA Dataset EULA."
)

def is_mdd(subj_id: str) -> int:
    return 1 if subj_id.startswith("0201") else 0

n_mdd = sum(is_mdd(s) for s in SUBJECT_IDS)
n_hc = len(SUBJECT_IDS) - n_mdd
print(f"[INFO] subjects={len(SUBJECT_IDS)} (MDD={n_mdd}, HC={n_hc})")
print(f"[INFO] Majority baseline (always HC): {n_hc/len(SUBJECT_IDS)*100:.2f}%")


# ======================
# Load .mat (robust key selection)
# ======================
def load_eeg_mat(path: str, n_channels: int = 128) -> np.ndarray:
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    keys = [k for k in mat.keys() if not k.startswith("__")]

    candidates = []
    for k in keys:
        v = mat[k]
        try:
            arr = np.asarray(v)
        except Exception:
            continue
        if arr.ndim != 2:
            continue
        if not np.issubdtype(arr.dtype, np.number):
            continue
        candidates.append((k, arr))

    if not candidates:
        raise RuntimeError(f"No 2D numeric EEG-like array found in {path} keys={keys}")

    _, data = max(candidates, key=lambda kv: kv[1].size)
    data = np.asarray(data, dtype=np.float32)

    # normalize orientation to (C,T)
    if data.shape[0] == n_channels + 1:
        data = data[:n_channels, :]
    elif data.shape[1] == n_channels + 1:
        data = data.T[:n_channels, :]
    elif data.shape[0] == n_channels:
        pass
    elif data.shape[1] == n_channels:
        data = data.T
    else:
        if data.shape[0] > data.shape[1]:
            data = data[:n_channels, :]
        else:
            data = data.T[:n_channels, :]
    return data


# ======================
# Locate .mat files and map subjects
# ======================
all_mat_files = glob.glob(os.path.join(RAW_ROOT, "**", "*.mat"), recursive=True)
print("[INFO] Found", len(all_mat_files), ".mat files under", RAW_ROOT)
if len(all_mat_files) == 0:
    raise RuntimeError("No .mat files found. Check RAW_ROOT / ZIP_PATH.")

def pick_best_match(paths: List[str]) -> str:
    if len(paths) == 1:
        return paths[0]
    return sorted(paths, key=lambda p: os.path.getsize(p), reverse=True)[0]

subj_to_path: Dict[str, str] = {}
for sid in SUBJECT_IDS:
    matches = [p for p in all_mat_files if sid in os.path.basename(p)]
    if not matches:
        matches = [p for p in all_mat_files if sid in p]
    if not matches:
        raise RuntimeError(f"Could not find .mat for subject {sid}")
    subj_to_path[sid] = pick_best_match(matches)
print("[INFO] Mapped", len(subj_to_path), "subjects to .mat files.")


# ======================
# DE features
# ======================
SEG_LEN = cfg.FS * cfg.SEG_SECONDS
BANDS = list(cfg.BANDS)
N_BANDS = len(BANDS)

def design_band_filters(fs: int = cfg.FS):
    nyq = fs / 2.0
    filters = []
    for low, high in BANDS:
        b, a = butter(4, [low / nyq, high / nyq], btype="bandpass")
        filters.append((b, a))
    return filters

BAND_FILTERS = design_band_filters()

def zscore_de_subjectwise(de: np.ndarray) -> np.ndarray:
    mu = de.mean(axis=0, keepdims=True)
    sd = de.std(axis=0, keepdims=True) + 1e-6
    return (de - mu) / sd

def compute_de_for_subject(eeg: np.ndarray) -> np.ndarray:
    C, T = eeg.shape
    max_n_seg = T // SEG_LEN
    if max_n_seg < 1:
        raise RuntimeError(f"Not enough length: T={T}")

    if max_n_seg >= cfg.N_SEGMENTS:
        start_seg = 0 if cfg.CROP_MODE == "first" else int(np.random.RandomState(cfg.SEED).randint(0, max_n_seg - cfg.N_SEGMENTS + 1))
        start = start_seg * SEG_LEN
        eeg = eeg[:, start: start + cfg.N_SEGMENTS * SEG_LEN]
        n_seg = cfg.N_SEGMENTS
    else:
        n_seg = max_n_seg
        eeg = eeg[:, : n_seg * SEG_LEN]

    de = np.empty((n_seg, C, N_BANDS), dtype=np.float32)
    for bi, (b, a) in enumerate(BAND_FILTERS):
        filtered = filtfilt(b, a, eeg, axis=1)
        reshaped = filtered.reshape(C, n_seg, SEG_LEN)
        var = reshaped.var(axis=2, ddof=0) + 1e-8
        de_band = 0.5 * np.log(2 * math.pi * math.e * var)
        de[:, :, bi] = de_band.T

    if n_seg < cfg.N_SEGMENTS:
        pad = np.repeat(de[-1:], repeats=(cfg.N_SEGMENTS - n_seg), axis=0)
        de = np.concatenate([de, pad], axis=0)
    return de[:cfg.N_SEGMENTS]


# ======================
# Adjacency + Chebyshev polynomials
# ======================
def compute_adj_from_de(de: np.ndarray, phi: float, zero_diag: bool) -> np.ndarray:
    O, C, _F = de.shape
    feat = de.transpose(1, 0, 2).reshape(C, -1)
    corr = np.corrcoef(feat)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    A = (np.abs(corr) >= phi).astype(np.float32)
    np.fill_diagonal(A, 0.0 if zero_diag else 1.0)
    return A

def compute_chebyshev_polynomials(A: np.ndarray, K: int) -> np.ndarray:
    C = A.shape[0]
    D = np.diag(A.sum(axis=1))
    L = D - A
    L = (L + L.T) / 2.0
    eigvals = np.linalg.eigvalsh(L)
    lmax = float(np.max(eigvals).real)
    if lmax < 1e-6:
        L_tilde = np.eye(C, dtype=np.float32)
    else:
        L_tilde = (2.0 * L / lmax) - np.eye(C, dtype=np.float32)

    T_k = [np.eye(C, dtype=np.float32)]
    if K > 1:
        T_k.append(L_tilde.astype(np.float32))
    for k in range(2, K):
        T_k.append(2 * L_tilde @ T_k[-1] - T_k[-2])
    return np.stack(T_k, axis=0)


# ======================
# MMD utilities (RBF)
# ======================
def pairwise_sq_dists(X: np.ndarray, Y: np.ndarray) -> np.ndarray:
    X = X.astype(np.float64, copy=False)
    Y = Y.astype(np.float64, copy=False)
    XX = np.sum(X ** 2, axis=1, keepdims=True)
    YY = np.sum(Y ** 2, axis=1, keepdims=True).T
    d = XX + YY - 2.0 * (X @ Y.T)
    return np.maximum(d, 0.0)

def estimate_sigma_median(all_features: np.ndarray, max_samples: int, seed: int) -> float:
    rng = np.random.RandomState(seed)
    n_total = all_features.shape[0]
    n = min(max_samples, n_total)
    idx = rng.choice(n_total, size=n, replace=False)
    sub = all_features[idx]
    d = pairwise_sq_dists(sub, sub)
    tri = d[np.triu_indices_from(d, k=1)]
    tri = tri[tri > 0]
    med = np.median(tri) if tri.size > 0 else 1.0
    sigma = math.sqrt(0.5 * med) if med > 0 else 1.0
    return float(sigma)

def parse_bandwidth(bw_cfg) -> float:
    if isinstance(bw_cfg, (int, float)):
        return float(bw_cfg)
    if isinstance(bw_cfg, str):
        s = bw_cfg.strip().lower()
        if s == "median":
            return float("nan")
        return float(s)
    return float("nan")

def mmd_rbf(X: np.ndarray, Y: np.ndarray, sigma: float, estimator: str = "biased") -> float:
    sigma = float(max(sigma, 1e-6))
    gamma = 1.0 / (2.0 * sigma * sigma)
    Kxx = np.exp(-gamma * pairwise_sq_dists(X, X))
    Kyy = np.exp(-gamma * pairwise_sq_dists(Y, Y))
    Kxy = np.exp(-gamma * pairwise_sq_dists(X, Y))

    n = X.shape[0]
    m = Y.shape[0]
    if estimator.lower() == "biased":
        val = float(Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean())
        return max(val, 0.0)

    if n > 1:
        Kxx2 = Kxx.copy()
        np.fill_diagonal(Kxx2, 0.0)
        term_x = Kxx2.sum() / (n * (n - 1))
    else:
        term_x = 0.0

    if m > 1:
        Kyy2 = Kyy.copy()
        np.fill_diagonal(Kyy2, 0.0)
        term_y = Kyy2.sum() / (m * (m - 1))
    else:
        term_y = 0.0

    term_xy = Kxy.mean()
    val = float(term_x + term_y - 2.0 * term_xy)
    return max(val, 0.0)


# ======================
# SSP clustering
# ======================
def run_ssp_iterative(
    features_list: List[np.ndarray],
    num_domains: int,
    bandwidth_cfg="median",
    estimator: str = "biased",
    weighted_merge: bool = False,
    seed: int = 0,
):
    S = len(features_list)
    if S < num_domains:
        raise ValueError(f"SSP: S({S}) < num_domains({num_domains})")

    bw = parse_bandwidth(bandwidth_cfg)
    if np.isnan(bw):
        all_concat = np.concatenate(features_list, axis=0)
        sigma = estimate_sigma_median(all_concat, max_samples=cfg.SIGMA_MEDIAN_MAX_SAMPLES, seed=seed)
    else:
        sigma = float(bw)

    cluster_feats = [f.copy() for f in features_list]
    cluster_members = [[i] for i in range(S)]

    n = S
    dist = np.full((n, n), np.inf, dtype=np.float64)
    for i in range(n):
        Xi = cluster_feats[i]
        for j in range(i + 1, n):
            Xj = cluster_feats[j]
            d = mmd_rbf(Xi, Xj, sigma, estimator=estimator)
            dist[i, j] = dist[j, i] = d

    while len(cluster_feats) > num_domains:
        i, j = np.unravel_index(np.argmin(dist), dist.shape)
        if i == j or not np.isfinite(dist[i, j]):
            break

        fi, fj = cluster_feats[i], cluster_feats[j]
        mi, mj = cluster_members[i], cluster_members[j]

        if weighted_merge:
            wi, wj = len(mi), len(mj)
            new_feat = (wi * fi + wj * fj) / float(wi + wj)
        else:
            new_feat = (fi + fj) / 2.0

        new_members = mi + mj

        keep = [k for k in range(len(cluster_feats)) if k not in (i, j)]
        new_cluster_feats = [cluster_feats[k] for k in keep] + [new_feat]
        new_cluster_members = [cluster_members[k] for k in keep] + [new_members]

        n_new = len(new_cluster_feats)
        new_dist = np.full((n_new, n_new), np.inf, dtype=np.float64)

        old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(keep)}
        for a_old in keep:
            a_new = old_to_new[a_old]
            for b_old in keep:
                b_new = old_to_new[b_old]
                if a_old == b_old:
                    continue
                new_dist[a_new, b_new] = dist[a_old, b_old]

        new_idx = n_new - 1
        Xnew = new_feat
        for k_new in range(n_new - 1):
            Xk = new_cluster_feats[k_new]
            d = mmd_rbf(Xk, Xnew, sigma, estimator=estimator)
            new_dist[k_new, new_idx] = new_dist[new_idx, k_new] = d

        cluster_feats = new_cluster_feats
        cluster_members = new_cluster_members
        dist = new_dist

    domain_labels = np.empty(S, dtype=np.int64)
    for d, members in enumerate(cluster_members):
        for sidx in members:
            domain_labels[sidx] = d
    return domain_labels, cluster_members, sigma


# ======================
# Model
# ======================
class ChebGCNLayer(nn.Module):
    def __init__(self, K: int, in_channels: int, out_channels: int, bias: bool = True):
        super().__init__()
        self.K = K
        self.theta = nn.Parameter(torch.Tensor(K, in_channels, out_channels))
        self.bias = nn.Parameter(torch.Tensor(out_channels)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.theta)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor, cheb: torch.Tensor) -> torch.Tensor:
        B, C, Fin = x.shape
        K, Fin2, Fout = self.theta.shape
        assert K == self.K and Fin2 == Fin

        if cheb.dim() == 3:
            cheb = cheb.unsqueeze(0).expand(B, -1, -1, -1)

        out = x.new_zeros((B, C, Fout))
        for k in range(K):
            Tx = torch.bmm(cheb[:, k], x)
            out = out + torch.matmul(Tx, self.theta[k])
        if self.bias is not None:
            out = out + self.bias
        return out


class GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd: float):
        ctx.lambd = float(lambd)
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return GradReverseFn.apply(x, lambd)


class SSPAGCN(nn.Module):
    def __init__(self, n_channels: int, n_bands: int, K: int,
                 gcn_hidden: int, gcn_out: int, num_domains: int,
                 dropout: float, pooling: str):
        super().__init__()
        self.pooling = pooling
        self.n_channels = n_channels
        self.gcn_out = gcn_out

        self.atten = nn.Parameter(torch.ones(n_channels, n_bands))

        self.gcn1 = ChebGCNLayer(K, n_bands, gcn_hidden)
        self.bn1 = nn.BatchNorm1d(gcn_hidden)

        self.gcn2 = ChebGCNLayer(K, gcn_hidden, gcn_out)
        self.bn2 = nn.BatchNorm1d(gcn_out)

        self.dropout = nn.Dropout(dropout)

        # Shared bottleneck for flatten pooling (fix)
        if pooling == "flatten":
            self.pool_proj = nn.Sequential(
                nn.Linear(n_channels * gcn_out, gcn_out),
                nn.ReLU(),
            )
        else:
            self.pool_proj = nn.Identity()

        feat_dim = gcn_out

        self.fc_label = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )
        self.fc_domain = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_domains),
        )

    def forward(self, x: torch.Tensor, cheb: torch.Tensor, lambd: float = 1.0):
        x = F.relu(x * self.atten)

        h = self.gcn1(x, cheb)
        B, C, H = h.shape
        h = self.bn1(h.view(B * C, H))
        h = self.dropout(F.relu(h)).view(B, C, H)

        h = self.gcn2(h, cheb)
        B, C, Fout = h.shape
        h = self.bn2(h.view(B * C, Fout))
        h = self.dropout(F.relu(h)).view(B, C, Fout)

        if self.pooling == "mean":
            feat = h.mean(dim=1)              # (B, Fout)
        elif self.pooling == "flatten":
            feat = h.reshape(B, C * Fout)     # (B, C*Fout)
        else:
            raise ValueError(self.pooling)

        feat = self.pool_proj(feat)           # (B, Fout)

        logp_label = F.log_softmax(self.fc_label(feat), dim=1)
        feat_rev = grad_reverse(feat, lambd)
        logp_domain = F.log_softmax(self.fc_domain(feat_rev), dim=1)
        return logp_label, logp_domain


# backbone-only regularization (fix)
def l1_l2_regularization_backbone(model: nn.Module, alpha: float, beta: float) -> torch.Tensor:
    l1 = torch.tensor(0., device=device)
    l2_sq = torch.tensor(0., device=device)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        # weights only (exclude biases/BN)
        if p.ndim <= 1:
            continue
        # include only backbone: atten + gcn thetas
        if not (name.startswith("atten") or name.startswith("gcn1") or name.startswith("gcn2")):
            continue
        l1 = l1 + p.abs().sum()
        l2_sq = l2_sq + (p ** 2).sum()
    l2 = torch.sqrt(l2_sq + 1e-12)
    return alpha * l1 + beta * l2


# ======================
# Dataset (TRAIN/EVAL always use 150 segments)
# ======================
class EEGSegmentDataset(Dataset):
    def __init__(self, subject_indices: List[int], de_list, cheb_list, soft_labels, domain_labels):
        self.samples = []
        self.de_list = de_list
        self.cheb_list = cheb_list
        self.soft_labels = soft_labels
        self.domain_labels = domain_labels
        for si in subject_indices:
            for ti in range(cfg.N_SEGMENTS):
                self.samples.append((si, ti))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        si, ti = self.samples[idx]
        x = torch.from_numpy(self.de_list[si][ti]).float()      # (C,F)
        cheb = torch.from_numpy(self.cheb_list[si]).float()     # (K,C,C)
        y_soft = torch.from_numpy(self.soft_labels[si]).float() # (2,)
        d = int(self.domain_labels[si]) if self.domain_labels[si] >= 0 else 0
        y_hard = int(is_mdd(SUBJECT_IDS[si]))
        return x, cheb, y_soft, torch.tensor(d, dtype=torch.long), torch.tensor(y_hard, dtype=torch.long)


# ======================
# Metrics + GRL schedule
# ======================
def compute_metrics_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(np.int64)
    y_pred = np.asarray(y_pred).astype(np.int64)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    acc = (tp + tn) / max(1, (tp + tn + fp + fn))
    rec = tp / max(1, (tp + fn))                 # sensitivity
    spec = tn / max(1, (tn + fp))                # specificity
    pre = tp / max(1, (tp + fp))
    f1 = 2 * pre * rec / max(1e-8, (pre + rec))
    bacc = 0.5 * (rec + spec)
    return acc, bacc, rec, spec, pre, f1, (tp, tn, fp, fn)

def grl_lambda(step: int, total_steps: int) -> float:
    if cfg.GRL_MODE == "constant":
        return float(cfg.GRL_LAMBDA)
    p = step / float(max(1, total_steps))
    return float(2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)


# ======================
# Precompute DE + cheb + labels + SSP features
# ======================
print("[INFO] Precomputing DE + chebyshev (both ZERO_DIAG options) ...")
de_list = []
soft_label_list = []
hard_label_list = []
cheb_cache = {True: [], False: []}
mmd_feature_list_full = []

for sid in SUBJECT_IDS:
    eeg = load_eeg_mat(subj_to_path[sid], n_channels=cfg.N_CHANNELS)
    de = compute_de_for_subject(eeg)  # (150,128,5)

    de_model = zscore_de_subjectwise(de) if cfg.ZSCORE_DE_FOR_MODEL else de
    de_ssp = zscore_de_subjectwise(de) if cfg.ZSCORE_DE_FOR_SSP else de

    for zd in (True, False):
        A = compute_adj_from_de(de_model, phi=cfg.PHI, zero_diag=zd)
        cheb = compute_chebyshev_polynomials(A, K=cfg.CHEB_K)
        cheb_cache[zd].append(cheb.astype(np.float32))

    score = float(PHQ_SCORES[sid])
    soft = np.array([score / cfg.PHQ_MAX, 1.0 - score / cfg.PHQ_MAX], dtype=np.float32)
    hard = is_mdd(sid)

    de_list.append(de_model.astype(np.float32))
    soft_label_list.append(soft)
    hard_label_list.append(hard)
    mmd_feature_list_full.append(de_ssp.reshape(cfg.N_SEGMENTS, -1).astype(np.float32))

print("[INFO] Done precompute for", len(SUBJECT_IDS), "subjects.")


# SSP features subsample only for MMD distances
ssp_k = int(min(cfg.N_SEGMENTS, max(1, SSP_MMD_SEGMENTS if FAST_MINI else cfg.N_SEGMENTS)))
ssp_idx = np.arange(ssp_k, dtype=np.int64)
mmd_feature_list = [f[ssp_idx] for f in mmd_feature_list_full]
print(f"[INFO] SSP/MMD segments used per subject: {len(ssp_idx)} / {cfg.N_SEGMENTS}")
print(f"[INFO] TRAIN/EVAL segments used per subject: {cfg.N_SEGMENTS} / {cfg.N_SEGMENTS} (paper-aligned)")


# ======================
# Domain label providers with caching
# ======================
num_subjects = len(SUBJECT_IDS)

paper_domain_labels: Optional[np.ndarray] = None
paper_sigma: Optional[float] = None
strict_domain_cache: Dict[int, np.ndarray] = {}

def get_domain_labels(ssp_mode: str, test_idx: int) -> np.ndarray:
    global paper_domain_labels, paper_sigma, strict_domain_cache
    assert ssp_mode in ("paper", "strict")

    if ssp_mode == "paper":
        if paper_domain_labels is None:
            print("[SSP] SSP_MODE=paper: computing SSP on ALL subjects once...")
            dl, _clusters, sigma_used = run_ssp_iterative(
                mmd_feature_list,
                num_domains=cfg.DN,
                bandwidth_cfg=cfg.MMD_BANDWIDTH,
                estimator=cfg.MMD_ESTIMATOR,
                weighted_merge=cfg.WEIGHTED_MERGE,
                seed=cfg.SEED,
            )
            paper_domain_labels = dl
            paper_sigma = float(sigma_used)
            counts = {d: int(np.sum(dl == d)) for d in range(cfg.DN)}
            print("[SSP] done. sigma=%.4f domain_counts=%s" % (paper_sigma, counts))
        return paper_domain_labels.copy()

    # strict
    if test_idx in strict_domain_cache:
        return strict_domain_cache[test_idx].copy()

    train_idx = [i for i in range(num_subjects) if i != test_idx]
    feats_train = [mmd_feature_list[i] for i in train_idx]
    dl_train, _clusters, _sigma = run_ssp_iterative(
        feats_train,
        num_domains=cfg.DN,
        bandwidth_cfg=cfg.MMD_BANDWIDTH,
        estimator=cfg.MMD_ESTIMATOR,
        weighted_merge=cfg.WEIGHTED_MERGE,
        seed=cfg.SEED,
    )
    domain_labels = np.full(num_subjects, -1, dtype=np.int64)
    for local_i, global_i in enumerate(train_idx):
        domain_labels[global_i] = int(dl_train[local_i])
    domain_labels[test_idx] = 0
    strict_domain_cache[test_idx] = domain_labels
    return domain_labels.copy()


# ======================
# LOSO runner
# ======================
def run_loso_single_config(ssp_mode: str, zero_diag: bool, pooling: str) -> Dict[str, object]:
    assert ssp_mode in ("paper","strict")
    assert pooling in ("mean","flatten")

    y_true = np.array(hard_label_list, dtype=np.int64)
    y_pred = np.zeros(num_subjects, dtype=np.int64)

    cheb_list = cheb_cache[zero_diag]

    test_indices = list(range(num_subjects))
    if LIMIT_N_TEST_SUBJECTS and LIMIT_N_TEST_SUBJECTS > 0:
        test_indices = test_indices[: int(LIMIT_N_TEST_SUBJECTS)]

    for fold_i, test_idx in enumerate(test_indices, start=1):
        seed_everything(cfg.SEED + test_idx)

        dom = get_domain_labels(ssp_mode, test_idx)
        train_idx = [i for i in range(num_subjects) if i != test_idx]

        train_ds = EEGSegmentDataset(train_idx, de_list, cheb_list, soft_label_list, dom)
        test_ds  = EEGSegmentDataset([test_idx], de_list, cheb_list, soft_label_list, dom)

        torch_gen = torch.Generator()
        torch_gen.manual_seed(cfg.SEED + test_idx)

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0, generator=torch_gen)
        test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

        model = SSPAGCN(
            n_channels=cfg.N_CHANNELS,
            n_bands=N_BANDS,
            K=cfg.CHEB_K,
            gcn_hidden=cfg.GCN_HIDDEN,
            gcn_out=cfg.GCN_OUT,
            num_domains=cfg.DN,
            dropout=cfg.DROPOUT,
            pooling=pooling,
        ).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=0.0)

        model.train()
        total_steps = cfg.EPOCHS * max(1, len(train_loader))
        step = 0
        for _ in range(cfg.EPOCHS):
            for x, cheb, y_soft, d, _y_hard in train_loader:
                x = x.to(device)
                cheb = cheb.to(device)
                y_soft = y_soft.to(device)
                d = d.to(device)

                lambd = grl_lambda(step, total_steps)
                step += 1

                opt.zero_grad()
                log_pl, log_pd = model(x, cheb, lambd=lambd)

                loss_label = -(y_soft * log_pl).sum(dim=1).mean()
                loss_domain = F.nll_loss(log_pd, d) * float(cfg.DOMAIN_LOSS_WEIGHT)
                loss_reg = l1_l2_regularization_backbone(model, cfg.ALPHA_L1, cfg.BETA_L2)

                loss = loss_label + loss_domain + loss_reg
                loss.backward()

                if cfg.CLIP_GRAD_NORM and cfg.CLIP_GRAD_NORM > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD_NORM)
                opt.step()

        # evaluate subject by mean prob over 150 segments
        model.eval()
        seg_probs = []
        with torch.no_grad():
            for x, cheb, _y_soft, _d, _y_hard in test_loader:
                x = x.to(device)
                cheb = cheb.to(device)
                log_pl, _ = model(x, cheb, lambd=0.0)
                seg_probs.append(log_pl.exp().cpu().numpy())
        seg_probs = np.concatenate(seg_probs, axis=0)  # (150,2)
        mean_prob = seg_probs.mean(axis=0)             # [Dep,Nor]
        pred_mdd = int(mean_prob[0] >= mean_prob[1])
        y_pred[test_idx] = pred_mdd

        if VERBOSE_FOLDS:
            sid = SUBJECT_IDS[test_idx]
            print(f"[fold {fold_i:02d}/{len(test_indices)}] test={sid} pred={'MDD' if pred_mdd==1 else 'HC'} "
                  f"(Dep={mean_prob[0]:.3f}, Nor={mean_prob[1]:.3f})")

    y_true_eval = y_true[test_indices]
    y_pred_eval = y_pred[test_indices]
    acc, bacc, rec, spec, pre, f1, cm = compute_metrics_binary(y_true_eval, y_pred_eval)

    pred_pos = int(np.sum(y_pred_eval == 1))
    out = {
        "SSP_MODE": ssp_mode,
        "ZERO_DIAG": bool(zero_diag),
        "POOLING": pooling,
        "EPOCHS": int(cfg.EPOCHS),
        "BATCH": int(cfg.BATCH_SIZE),
        "SSP_MMD_SEGS": int(len(ssp_idx)),
        "N_TEST_SUBJECTS": int(len(test_indices)),
        "DN": int(cfg.DN),
        "Acc_subj": float(acc),
        "BAcc_subj": float(bacc),
        "Recall_subj": float(rec),
        "Spec_subj": float(spec),
        "Precision_subj": float(pre),
        "F1_subj": float(f1),
        "TP": int(cm[0]), "TN": int(cm[1]), "FP": int(cm[2]), "FN": int(cm[3]),
        "Pred_MDD_count": int(pred_pos),
    }
    if ssp_mode == "paper" and paper_sigma is not None:
        out["SSP_sigma_global"] = float(paper_sigma)
    return out


# ======================
# Switch grid (3×2)
# ======================
grid = list(itertools.product(["paper","strict"], [True, False], ["mean","flatten"]))
if not RUN_SWEEP:
    grid = [("paper", True, "mean")]

print("\n[INFO] Running configs:")
for ssp_mode, zd, pool in grid:
    print("   - SSP_MODE=", ssp_mode, "| ZERO_DIAG=", zd, "| POOLING=", pool)

results = []
for i, (ssp_mode, zd, pool) in enumerate(grid, start=1):
    print(f"\n===== CONFIG {i}/{len(grid)}: SSP_MODE={ssp_mode}, ZERO_DIAG={zd}, POOLING={pool} =====")
    res = run_loso_single_config(ssp_mode, zd, pool)
    print(f"  -> Acc={res['Acc_subj']*100:.2f}%, BAcc={res['BAcc_subj']*100:.2f}%, "
          f"Recall={res['Recall_subj']*100:.2f}%, Spec={res['Spec_subj']*100:.2f}%, "
          f"Prec={res['Precision_subj']*100:.2f}%, F1={res['F1_subj']*100:.2f}% "
          f"(TP,TN,FP,FN)=({res['TP']},{res['TN']},{res['FP']},{res['FN']}) "
          f"| Pred_MDD={res['Pred_MDD_count']}")
    results.append(res)

df = pd.DataFrame(results).sort_values(["SSP_MODE","ZERO_DIAG","POOLING"]).reset_index(drop=True)

print("\n===== SUMMARY (subject-level, LOSO) =====")
cols = ["SSP_MODE","ZERO_DIAG","POOLING","EPOCHS","BATCH","SSP_MMD_SEGS","N_TEST_SUBJECTS",
        "Acc_subj","BAcc_subj","Recall_subj","Spec_subj","Precision_subj","F1_subj",
        "TP","TN","FP","FN","Pred_MDD_count"]
print(df[cols].to_string(index=False))

out_csv = "/content/sspagcn_switch_sweep_results_STABLE.csv"
df.to_csv(out_csv, index=False)
print("\n[INFO] Saved CSV:", out_csv)

[INFO] device: cuda
Mounted at /content/drive
[INFO] Unzipping MODMA zip (only once)...
[INFO] Unzip done.
[INFO] subjects=53 (MDD=24, HC=29)
[INFO] Majority baseline (always HC): 54.72%
[INFO] Found 53 .mat files under /content/modma_raw
[INFO] Mapped 53 subjects to .mat files.
[INFO] Precomputing DE + chebyshev (both ZERO_DIAG options) ...
[INFO] Done precompute for 53 subjects.
[INFO] SSP/MMD segments used per subject: 30 / 150
[INFO] TRAIN/EVAL segments used per subject: 150 / 150 (paper-aligned)

[INFO] Running configs:
   - SSP_MODE= paper | ZERO_DIAG= True | POOLING= mean
   - SSP_MODE= paper | ZERO_DIAG= True | POOLING= flatten
   - SSP_MODE= paper | ZERO_DIAG= False | POOLING= mean
   - SSP_MODE= paper | ZERO_DIAG= False | POOLING= flatten
   - SSP_MODE= strict | ZERO_DIAG= True | POOLING= mean
   - SSP_MODE= strict | ZERO_DIAG= True | POOLING= flatten
   - SSP_MODE= strict | ZERO_DIAG= False | POOLING= mean
   - SSP_MODE= strict | ZERO_DIAG= False | POOLING= flatten

===== CO

# GDN Switch-Sensitivity Mini-Study

In [ ]:
# ============================================================
# Generative Depression Discriminator (GDN) Switch-Sensitivity Mini-Study (MODMA EEG)
# 3 QA-critical switches × 2 levels = 8 configs
#
# Switches (paper-consistent ambiguities):
#   (S1) SIM_SIGNAL_FOR_TOPK : "raw" vs "filtered"          (Section 2.6.2)
#   (S2) HAMMING_MODE        : "paper_plus_2pi" vs "numpy_hamming" (Section 2.6.3)
#   (S3) THRESH_TUNE_LEVEL   : "segment" vs "subject"      (Section 2.6.5: "validation accuracy" unit is underdetermined)
#
# Output:
#   - segment-level metrics + subject-level metrics (majority vote)
#   - a results table (8 rows) + accuracy ranges, saved as CSV
#
# You MUST set ZIP_PATH (or DATA_ROOT) for your MODMA EEG files.
# ============================================================

import os, re, glob, random, warnings, sys, subprocess, itertools
from collections import defaultdict
import numpy as np

# ---- lightweight pip install (Colab-safe) ----
def _pip_install(pkg: str):
    try:
        __import__(pkg.split("==")[0].split(">=")[0].split("[")[0])
        return
    except Exception:
        pass
    print(f"[pip] installing {pkg} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip_install("pywavelets")
_pip_install("tqdm")
_pip_install("scikit-learn")
_pip_install("pandas")

import pywt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import confusion_matrix
import pandas as pd

# display() fallback
try:
    from IPython.display import display  # type: ignore
except Exception:
    def display(x):
        print(x)

# =========================
# User settings (EDIT HERE)
# =========================
SEED = 0

# If using Drive, put the zip in Drive and set ZIP_PATH accordingly
ZIP_PATH  = "/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015.zip"
DATA_ROOT = "/content/modma_lanzhou"  # extraction target (or existing folder)

# Dataset constants (paper-aligned)
FS = 250
SEG_LEN = 10 * FS          # 10 seconds (2500 samples)
N_CHANNELS = 128
K_SIM = 10                 # top-k similar channels

# Force max segments per subject (paper: max 8 for this method)
FORCE_MAX_SEGS_PER_SUBJECT = True
MAX_SEGS_PER_SUBJECT = 8

# Preprocessing (fixed here; you can add as additional switches if needed)
DEMEAN_PER_CHANNEL = True
ZSCORE_PER_CHANNEL = False

# FFT bandpass (paper-aligned)
FFT_LOW_HZ  = 4.0
FFT_HIGH_HZ = 14.0

# DWT (paper-aligned)
WAVELET_NAME = "db6"
DWT_MODE = "symmetric"

# Training hyperparams (mini by default)
FAST_MINI = True
EPOCHS = 6 if FAST_MINI else 20
LR = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 128 if FAST_MINI else 256
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[INFO] device:", DEVICE)

# Expected class counts for MODMA (used only for label auto-inference)
EXPECTED_MDD = 24
EXPECTED_HC  = 29

# =========================
# Reproducibility
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# =========================
# Colab Drive mount (optional)
# =========================
try:
    from google.colab import drive  # type: ignore
    if ZIP_PATH.startswith("/content/drive/"):
        drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# =========================
# Unzip helper
# =========================
def unzip_if_needed(zip_path: str, data_root: str):
    if os.path.isdir(data_root) and len(glob.glob(os.path.join(data_root, "**/*.mat"), recursive=True)) > 0:
        return
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(
            f"ZIP_PATH not found: {zip_path}\n"
            f"Either upload the zip to Colab/Drive and set ZIP_PATH, or set DATA_ROOT to an extracted folder."
        )
    os.makedirs(data_root, exist_ok=True)
    print("[INFO] Unzipping zip ->", data_root)
    subprocess.check_call(["bash","-lc", f'unzip -q -o "{zip_path}" -d "{data_root}"'])

unzip_if_needed(ZIP_PATH, DATA_ROOT)

# =========================
# Subject ID + label inference
# =========================
SUBJECT_PATTERNS = [
    re.compile(r"^\d{3,10}$"),               # pure digits folder
    re.compile(r"^(sub|s)\d{2,10}$", re.I),
    re.compile(r"^(subject)\d{2,10}$", re.I),
]
GENERIC_DIRS = {
    "mdd","hc","control","healthy","normal","depression","patient",
    "eeg","eegs","data","dataset","modma","raw"
}

def infer_subject_id(path: str) -> str:
    parts = os.path.normpath(path).split(os.sep)
    for i in range(len(parts)-2, -1, -1):
        name = parts[i]
        low = name.lower()
        if low in GENERIC_DIRS:
            continue
        if len(name) > 32:
            continue
        for pat in SUBJECT_PATTERNS:
            if pat.match(name):
                return name
    base = os.path.basename(path)
    m = re.findall(r"\d{4,10}", base)
    if m:
        return m[0]
    m2 = re.findall(r"\d+", base)
    if m2:
        return m2[0]
    return base

def label_from_path_or_id(path: str, sid: str):
    low = os.path.normpath(path).lower()
    if any(tok in low.split(os.sep) for tok in ["mdd","depression","patient"]):
        return 1
    if any(tok in low.split(os.sep) for tok in ["hc","healthy","control","normal"]):
        return 0
    # MODMA convention commonly used in MODMA papers: IDs starting with "0201" are MDD
    return 1 if str(sid).startswith("0201") else 0

def build_prefix_groups(subject_ids, prefix_len):
    groups = defaultdict(list)
    for s in subject_ids:
        d = re.sub(r"\D", "", str(s))
        pref = d[:prefix_len] if len(d) >= prefix_len else d
        groups[pref].append(s)
    return groups

def choose_auto_prefix_mapping(subject_ids,
                               expected_mdd=EXPECTED_MDD,
                               expected_hc=EXPECTED_HC,
                               prefix_lens=(4,5,6),
                               max_groups=64):
    subject_ids = list(subject_ids)
    for L in prefix_lens:
        groups = build_prefix_groups(subject_ids, L)
        if len(groups) > max_groups:
            continue
        group_items = [(k, len(v), v) for k,v in groups.items()]
        idxs = list(range(len(group_items)))
        found = None
        for r in range(1, min(18, len(group_items))+1):
            for comb in itertools.combinations(idxs, r):
                ssum = sum(group_items[i][1] for i in comb)
                if ssum == expected_mdd:
                    found = set(comb)
                    break
            if found is not None:
                break
        if found is None:
            continue
        mapping = {}
        for i,(k,sz,vals) in enumerate(group_items):
            lab = 1 if i in found else 0
            for sid in vals:
                mapping[sid] = lab
        n1 = sum(mapping[s]==1 for s in subject_ids)
        n0 = len(subject_ids) - n1
        if n1 == expected_mdd and n0 == expected_hc:
            return L, mapping
    return None, None

# =========================
# Robust .mat segment loader
# =========================
from scipy.io import loadmat

def _is_numeric_array(x):
    return isinstance(x, np.ndarray) and (np.issubdtype(x.dtype, np.number) or x.dtype == np.object_)

def _score_candidate(arr: np.ndarray):
    if not isinstance(arr, np.ndarray):
        return -1
    shp = arr.shape
    if arr.ndim < 2 or arr.ndim > 4:
        return -1
    score = 0
    if 128 in shp or 129 in shp: score += 10
    if SEG_LEN in shp: score += 10
    score += (arr.size / 1e6)
    return score

def load_subject_segments(path: str, seg_len=SEG_LEN):
    """
    Return segments as (n_seg, 128, seg_len) float32.
    """
    mat = loadmat(path, squeeze_me=False, struct_as_record=False)
    keys = [k for k in mat.keys() if not k.startswith("__")]

    candidates = []
    for k in keys:
        v = mat[k]
        if _is_numeric_array(v):
            arr = np.asarray(v)
            candidates.append((k, arr, _score_candidate(arr)))
    candidates = sorted(candidates, key=lambda x: x[2], reverse=True)
    if not candidates or candidates[0][2] < 0:
        raise ValueError(f"No usable numeric array in {path}. keys={keys}")

    best_key, data, _ = candidates[0]
    data = np.asarray(data, dtype=np.float32)
    segs = None

    if data.ndim == 2:
        a = data
        if a.shape[0] in (128,129):
            x = a
        elif a.shape[1] in (128,129):
            x = a.T
        else:
            x = a if abs(a.shape[0]-128) < abs(a.shape[1]-128) else a.T
        if x.shape[0] == 129:
            x = x[:128]
        T = x.shape[1]
        n_seg = T // seg_len
        x = x[:, :n_seg*seg_len]
        segs = x.reshape(128, n_seg, seg_len).transpose(1,0,2)

    elif data.ndim == 3:
        a = data
        if a.shape[1] in (128,129) and a.shape[2] == seg_len:
            x = a[:, :128, :] if a.shape[1] == 129 else a
            segs = x
        elif a.shape[2] in (128,129) and a.shape[1] == seg_len:
            x = a.transpose(0,2,1)
            x = x[:, :128, :] if x.shape[1] == 129 else x
            segs = x
        else:
            shp = a.shape
            ch_axis = 0 if shp[0] in (128,129) else (1 if shp[1] in (128,129) else (2 if shp[2] in (128,129) else None))
            if ch_axis is None:
                raise ValueError(f"3D array without 128/129 dim: {shp} in {path} (key={best_key})")
            x = np.moveaxis(a, ch_axis, 0)
            x = x[:128] if x.shape[0] == 129 else x
            x2 = x.reshape(128, -1)
            T = x2.shape[1]
            n_seg = T // seg_len
            x2 = x2[:, :n_seg*seg_len]
            segs = x2.reshape(128, n_seg, seg_len).transpose(1,0,2)

    elif data.ndim == 4:
        shp = data.shape
        ch_axis = next((ax for ax,sz in enumerate(shp) if sz in (128,129)), None)
        if ch_axis is None:
            raise ValueError(f"4D array without 128/129 dim: {shp} in {path} (key={best_key})")
        x = np.moveaxis(data, ch_axis, 1)
        x = x[:, :128, :, :] if x.shape[1] == 129 else x
        d0, ch, d2, d3 = x.shape
        x = x.reshape(d0, ch, d2*d3)
        x2 = x.transpose(1,0,2).reshape(ch, -1)
        T = x2.shape[1]
        n_seg = T // seg_len
        x2 = x2[:, :n_seg*seg_len]
        segs = x2.reshape(ch, n_seg, seg_len).transpose(1,0,2)

    else:
        raise ValueError(f"Unsupported ndim={data.ndim} in {path} (key={best_key})")

    if segs is None:
        raise ValueError(f"Failed to parse segments from {path} (key={best_key})")

    if FORCE_MAX_SEGS_PER_SUBJECT and segs.shape[0] > MAX_SEGS_PER_SUBJECT:
        segs = segs[:MAX_SEGS_PER_SUBJECT]
    return segs.astype(np.float32)

# =========================
# Similarity + filtering + wavelet
# =========================
def make_window(n: int, hamming_mode: str):
    t = np.arange(n, dtype=np.float32)
    if hamming_mode == "numpy_hamming":
        return np.hamming(n).astype(np.float32)
    if hamming_mode == "paper_plus_2pi":
        return (0.54 + 0.46*np.cos(2*np.pi*t/(n-1))).astype(np.float32)
    if hamming_mode == "paper_minus_2pi":
        return (0.54 - 0.46*np.cos(2*np.pi*t/(n-1))).astype(np.float32)
    if hamming_mode == "paper_literal":
        return (0.54 - 0.46*np.cos(t/(n-1))).astype(np.float32)
    raise ValueError("Unknown hamming_mode: " + str(hamming_mode))

def bandpass_fft(eeg_128xT: np.ndarray, hamming_mode: str, fs=FS, low=FFT_LOW_HZ, high=FFT_HIGH_HZ):
    ch, n = eeg_128xT.shape
    w = make_window(n, hamming_mode=hamming_mode)
    xw = eeg_128xT * w[None, :]
    X = np.fft.rfft(xw, axis=1)
    freqs = np.fft.rfftfreq(n, d=1.0/fs)
    mask = (freqs >= low) & (freqs <= high)
    X[:, ~mask] = 0.0
    y = np.fft.irfft(X, n=n, axis=1).astype(np.float32)
    y = y / (w[None, :] + 1e-6)
    return y.astype(np.float32)

W = pywt.Wavelet(WAVELET_NAME)
EXPECTED_WLEN = pywt.dwt_coeff_len(SEG_LEN, W.dec_len, mode=DWT_MODE)
print("[INFO] EXPECTED_WLEN =", EXPECTED_WLEN, "(db6, symmetric, seg_len=2500 => typically 1255)")

def compute_wavelet_coeffs_all(eeg_128xT: np.ndarray, wavelet=WAVELET_NAME):
    cA_list, cD_list = [], []
    for ch in range(eeg_128xT.shape[0]):
        cA, cD = pywt.dwt(eeg_128xT[ch], wavelet, mode=DWT_MODE)
        cA = cA.astype(np.float32); cD = cD.astype(np.float32)
        if cA.shape[0] != EXPECTED_WLEN:
            if cA.shape[0] > EXPECTED_WLEN:
                s = (cA.shape[0]-EXPECTED_WLEN)//2
                cA = cA[s:s+EXPECTED_WLEN]
                cD = cD[s:s+EXPECTED_WLEN]
            else:
                pad = EXPECTED_WLEN - cA.shape[0]
                cA = np.pad(cA, (0,pad))
                cD = np.pad(cD, (0,pad))
        cA_list.append(cA); cD_list.append(cD)
    return np.stack(cA_list, axis=0), np.stack(cD_list, axis=0)

def idwt_to_len(cA_1d: np.ndarray, cD_1d: np.ndarray, out_len=SEG_LEN, wavelet=WAVELET_NAME):
    rec = pywt.idwt(cA_1d, cD_1d, wavelet, mode=DWT_MODE)
    if rec.shape[0] >= out_len:
        return rec[:out_len].astype(np.float32)
    return np.pad(rec.astype(np.float32), (0, out_len - rec.shape[0]))

def compute_cosine_topk(eeg_128xT: np.ndarray, k=K_SIM):
    norms = np.linalg.norm(eeg_128xT, axis=1, keepdims=True) + 1e-8
    x = eeg_128xT / norms
    sim = x @ x.T
    np.fill_diagonal(sim, -np.inf)  # exclude self (standard for "most similar other channels")
    idx = np.argsort(-sim, axis=1)[:, :k]
    return idx.astype(np.int16)

# =========================
# Segment container (stores raw/filt similarity + active pointer)
# =========================
class SegmentInfo:
    __slots__ = ("index","subject_id","label","cA_all","cD_all","sim_raw","sim_filt","sim_idx")
    def __init__(self, index, subject_id, label, cA_all, cD_all, sim_raw, sim_filt):
        self.index = int(index)
        self.subject_id = str(subject_id)
        self.label = int(label)
        self.cA_all = cA_all
        self.cD_all = cD_all
        self.sim_raw = sim_raw
        self.sim_filt = sim_filt
        self.sim_idx = sim_raw  # default active sim

def activate_sim_mode(segments, sim_signal: str):
    if sim_signal not in ("raw","filtered"):
        raise ValueError("sim_signal must be 'raw' or 'filtered'")
    for seg in segments:
        seg.sim_idx = seg.sim_raw if sim_signal == "raw" else seg.sim_filt

# =========================
# Model (paper-aligned)
# =========================
class EncoderBlock(nn.Module):
    def __init__(self, channels: int, use_act: bool):
        super().__init__()
        self.bn = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=1)
        self.use_act = use_act
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        b = self.bn(x)
        y = self.conv1(b)
        if self.use_act: y = self.act(y)
        y = y + b
        y = self.conv2(y)
        if self.use_act: y = self.act(y)
        return y

class GDNEncoder(nn.Module):
    def __init__(self, in_channels, latent_dim=300, num_blocks=6, wavelet_len=EXPECTED_WLEN, use_act=False):
        super().__init__()
        self.blocks = nn.ModuleList([EncoderBlock(in_channels, use_act) for _ in range(num_blocks)])
        self.fc_out = nn.Linear(in_channels * wavelet_len, latent_dim)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        b,c,l = x.shape
        x = x.reshape(b, c*l)
        return self.fc_out(x)

class DecoderBlock(nn.Module):
    def __init__(self, dim: int, use_act: bool, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.use_act = use_act
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        var  = x.var(dim=1, unbiased=False, keepdim=True)
        nrm  = (x - mean) / torch.sqrt(var + self.eps)
        h = self.fc1(nrm)
        if self.use_act: h = self.act(h)
        y = self.fc2(h + nrm)
        if self.use_act: y = self.act(y)
        return y

class GDNDecoder(nn.Module):
    def __init__(self, latent_dim=300, out_len=EXPECTED_WLEN, num_blocks=5, use_act=False):
        super().__init__()
        self.blocks = nn.ModuleList([DecoderBlock(latent_dim, use_act) for _ in range(num_blocks)])
        self.fc_out = nn.Linear(latent_dim, out_len)

    def forward(self, z):
        x = z
        for blk in self.blocks:
            x = blk(x)
        return self.fc_out(x)

class GDNGenerator(nn.Module):
    def __init__(self, in_channels, latent_dim=300, out_len=EXPECTED_WLEN, use_act=False):
        super().__init__()
        self.enc_cA = GDNEncoder(in_channels=in_channels, latent_dim=latent_dim, wavelet_len=out_len, use_act=use_act)
        self.enc_cD = GDNEncoder(in_channels=in_channels, latent_dim=latent_dim, wavelet_len=out_len, use_act=use_act)
        self.w1 = nn.Parameter(torch.tensor(0.5))
        self.w2 = nn.Parameter(torch.tensor(0.5))
        self.dec_cA = GDNDecoder(latent_dim=latent_dim, out_len=out_len, use_act=use_act)
        self.dec_cD = GDNDecoder(latent_dim=latent_dim, out_len=out_len, use_act=use_act)

    def forward(self, ScA, ScD):
        zA = self.enc_cA(ScA)
        zD = self.enc_cD(ScD)
        z = self.w1 * zA + self.w2 * zD
        return self.dec_cA(z), self.dec_cD(z)

# =========================
# Dataset
# =========================
class GeneratorDataset(Dataset):
    def __init__(self, segments, seg_ids):
        self.segments = segments
        self.seg_ids = list(seg_ids)
        self.K = int(segments[self.seg_ids[0]].sim_idx.shape[1]) if len(self.seg_ids) else K_SIM

    def __len__(self):
        return len(self.seg_ids) * N_CHANNELS

    def __getitem__(self, idx):
        seg_local = idx // N_CHANNELS
        ch = idx % N_CHANNELS
        seg = self.segments[self.seg_ids[seg_local]]

        sim = seg.sim_idx[ch]          # (K,)
        ScA = seg.cA_all[sim]          # (K,Lw)
        ScD = seg.cD_all[sim]
        OcA = seg.cA_all[ch]           # (Lw,)
        OcD = seg.cD_all[ch]

        return (torch.from_numpy(ScA).float(),
                torch.from_numpy(ScD).float(),
                torch.from_numpy(OcA).float(),
                torch.from_numpy(OcD).float())

# =========================
# Train + evaluation
# =========================
def train_generator(model, loader, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for ep in range(1, epochs+1):
        model.train()
        total = 0.0
        nb = 0
        for ScA, ScD, OcA, OcD in loader:
            ScA, ScD = ScA.to(DEVICE), ScD.to(DEVICE)
            OcA, OcD = OcA.to(DEVICE), OcD.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pcA, pcD = model(ScA, ScD)
            loss = F.mse_loss(pcA, OcA) + F.mse_loss(pcD, OcD)
            loss.backward()
            opt.step()
            total += float(loss.item())
            nb += 1
        if (ep == 1) or (ep == epochs) or (ep % max(1, epochs//3) == 0):
            print(f"  [train] ep {ep:02d}/{epochs} loss={total/max(1,nb):.4f}")
    return model

@torch.no_grad()
def reconstruction_mse_time(model, seg: SegmentInfo, batch_size=128):
    model.eval()
    cA_all = seg.cA_all
    cD_all = seg.cD_all
    sim = seg.sim_idx  # (128,K)
    ScA_np = cA_all[sim]  # (128,K,Lw)
    ScD_np = cD_all[sim]
    ScA = torch.from_numpy(ScA_np).float().to(DEVICE)
    ScD = torch.from_numpy(ScD_np).float().to(DEVICE)

    errs = []
    for i in range(0, N_CHANNELS, batch_size):
        ScA_b = ScA[i:i+batch_size]
        ScD_b = ScD[i:i+batch_size]
        pcA, pcD = model(ScA_b, ScD_b)  # (B,Lw)
        pcA = pcA.detach().cpu().numpy()
        pcD = pcD.detach().cpu().numpy()
        for j in range(pcA.shape[0]):
            ch = i + j
            pred = idwt_to_len(pcA[j], pcD[j], out_len=SEG_LEN)
            true = idwt_to_len(cA_all[ch], cD_all[ch], out_len=SEG_LEN)
            errs.append(float(np.mean((pred - true)**2)))
    return np.array(errs, dtype=np.float32)

@torch.no_grad()
def evaluate_segments(gen_mdd, gen_hc, segments, seg_ids):
    out = []
    for seg_id in seg_ids:
        seg = segments[seg_id]
        e_mdd = reconstruction_mse_time(gen_mdd, seg)
        e_hc  = reconstruction_mse_time(gen_hc,  seg)
        n_mdd = int((e_mdd < e_hc).sum())
        out.append({
            "seg_id": int(seg_id),
            "subject_id": seg.subject_id,
            "true_label": int(seg.label),
            "n_mdd_elec": n_mdd,
        })
    return out

def metrics_segment(results, n0: int):
    y = np.array([r["true_label"] for r in results], dtype=int)
    n = np.array([r["n_mdd_elec"] for r in results], dtype=int)
    yp = (n > n0).astype(int)
    cm = confusion_matrix(y, yp, labels=[1,0])  # [[TP,FN],[FP,TN]]
    tp = int(cm[0,0]); fn = int(cm[0,1]); fp = int(cm[1,0]); tn = int(cm[1,1])
    acc  = (tp+tn)/max(1,len(y))
    sens = tp/max(1,(tp+fn))
    spec = tn/max(1,(tn+fp))
    bal  = 0.5*(sens+spec)
    return {"acc":acc,"sens":sens,"spec":spec,"bal_acc":bal,"tp":tp,"tn":tn,"fp":fp,"fn":fn}

def subject_level_from_results(results, n0: int):
    by_subj = defaultdict(list)
    for r in results:
        pred = 1 if r["n_mdd_elec"] > n0 else 0
        by_subj[r["subject_id"]].append((pred, r["true_label"]))
    y_true = []
    y_pred = []
    for sid, items in by_subj.items():
        preds = [p for p,_ in items]
        true  = items[0][1]
        maj = 1 if sum(preds) >= (len(preds)/2.0) else 0
        y_true.append(true)
        y_pred.append(maj)
    cm = confusion_matrix(y_true, y_pred, labels=[1,0])  # [[TP,FN],[FP,TN]]
    tp = int(cm[0,0]); fn = int(cm[0,1]); fp = int(cm[1,0]); tn = int(cm[1,1])
    acc  = (tp+tn)/max(1,len(y_true))
    return acc, {"tp":tp,"tn":tn,"fp":fp,"fn":fn}

def find_best_threshold_segment(results):
    best = (-1.0, None)
    for n0 in range(0, N_CHANNELS+1):
        m = metrics_segment(results, n0)
        if m["acc"] > best[0]:
            best = (m["acc"], n0)
    return int(best[1]), float(best[0])

def find_best_threshold_subject(results):
    best = (-1.0, None)
    for n0 in range(0, N_CHANNELS+1):
        acc, _ = subject_level_from_results(results, n0)
        if acc > best[0]:
            best = (acc, n0)
    return int(best[1]), float(best[0])

# =========================
# Build file list + labels + split lists
# =========================
mat_files = glob.glob(os.path.join(DATA_ROOT, "**/*.mat"), recursive=True)
if len(mat_files) == 0:
    raise RuntimeError(f"No .mat files found under DATA_ROOT={DATA_ROOT}")

subj_files = defaultdict(list)
for p in mat_files:
    sid = infer_subject_id(p)
    subj_files[sid].append(p)

subject_ids = sorted(subj_files.keys())
print("[INFO] Unique subject IDs:", len(subject_ids))

subject_label = {}
for sid, files in subj_files.items():
    labs = [label_from_path_or_id(p, sid) for p in files]
    lab = 1 if sum(labs) >= (len(labs)/2.0) else 0
    subject_label[sid] = lab

n_mdd = sum(subject_label[s]==1 for s in subject_ids)
n_hc  = len(subject_ids) - n_mdd
print(f"[INFO] label heuristic counts: MDD={n_mdd}, HC={n_hc}")

if (n_mdd, n_hc) != (EXPECTED_MDD, EXPECTED_HC):
    print("[WARN] counts != expected MODMA (24/29). Trying auto-prefix mapping ...")
    pref_len, mapping = choose_auto_prefix_mapping(subject_ids)
    if mapping is not None:
        subject_label = mapping
        n_mdd = sum(subject_label[s]==1 for s in subject_ids)
        n_hc  = len(subject_ids) - n_mdd
        print(f"[INFO] auto-prefix mapping selected prefix_len={pref_len} -> MDD={n_mdd}, HC={n_hc}")
    else:
        print("[WARN] auto-prefix mapping failed. Continuing with heuristic labels.")

def _first_int(s):
    m = re.findall(r"\d+", str(s))
    return int(m[0]) if m else 10**18

ordered_mdd = sorted([s for s in subject_ids if subject_label[s]==1], key=_first_int)
ordered_hc  = sorted([s for s in subject_ids if subject_label[s]==0], key=_first_int)

def split_subjects(subj_list):
    return subj_list[:15], subj_list[15:20], subj_list[20:]

mdd_tr, mdd_va, mdd_te = split_subjects(ordered_mdd)
hc_tr,  hc_va,  hc_te  = split_subjects(ordered_hc)
print("[INFO] subjects train/val/test:",
      len(mdd_tr), len(mdd_va), len(mdd_te), "|", len(hc_tr), len(hc_va), len(hc_te))

# =========================
# Precompute segments for each HAMMING_MODE level
# =========================
HAMMING_LEVELS = ["paper_plus_2pi", "numpy_hamming"]  # S2 levels

segments_by_hamming = {}
subject_to_seg_ids_ref = None

print("\n[INFO] Precomputing per-segment features for each Hamming mode ...")
for hmode in HAMMING_LEVELS:
    print(f"\n--- Hamming mode = {hmode} ---")
    segments = []
    subject_to_seg_ids = defaultdict(list)
    seg_counter = 0

    for sid in tqdm(ordered_mdd + ordered_hc):
        y = subject_label[sid]
        files = sorted(subj_files[sid])

        subj_seg_list = []
        for path in files:
            try:
                segs = load_subject_segments(path)
                if segs.shape[0] > 0:
                    subj_seg_list.append(segs)
            except Exception as e:
                warnings.warn(f"Failed to load segments for {sid} file={os.path.basename(path)}: {e}")

        if len(subj_seg_list) == 0:
            warnings.warn(f"No segments found for subject {sid}")
            continue

        subj_segs = np.concatenate(subj_seg_list, axis=0)  # (n_seg,128,2500)
        if FORCE_MAX_SEGS_PER_SUBJECT and subj_segs.shape[0] > MAX_SEGS_PER_SUBJECT:
            subj_segs = subj_segs[:MAX_SEGS_PER_SUBJECT]
        if FORCE_MAX_SEGS_PER_SUBJECT and subj_segs.shape[0] < MAX_SEGS_PER_SUBJECT:
            warnings.warn(f"Subject {sid}: only {subj_segs.shape[0]} segments (max={MAX_SEGS_PER_SUBJECT}).")

        for i in range(subj_segs.shape[0]):
            raw = subj_segs[i].astype(np.float32)

            if DEMEAN_PER_CHANNEL:
                raw = raw - raw.mean(axis=1, keepdims=True)
            if ZSCORE_PER_CHANNEL:
                mu = raw.mean(axis=1, keepdims=True)
                sd = raw.std(axis=1, keepdims=True) + 1e-6
                raw = (raw - mu) / sd

            eeg_f = bandpass_fft(raw, hamming_mode=hmode)

            sim_raw  = compute_cosine_topk(raw,   k=K_SIM)
            sim_filt = compute_cosine_topk(eeg_f, k=K_SIM)

            cA_all, cD_all = compute_wavelet_coeffs_all(eeg_f)

            seg = SegmentInfo(
                index=seg_counter,
                subject_id=sid,
                label=y,
                cA_all=cA_all,
                cD_all=cD_all,
                sim_raw=sim_raw,
                sim_filt=sim_filt,
            )
            segments.append(seg)
            subject_to_seg_ids[sid].append(seg_counter)
            seg_counter += 1

    if subject_to_seg_ids_ref is None:
        subject_to_seg_ids_ref = {k: tuple(v) for k,v in subject_to_seg_ids.items()}
    else:
        cur = {k: tuple(v) for k,v in subject_to_seg_ids.items()}
        if cur != subject_to_seg_ids_ref:
            warnings.warn("Segment id mapping differs across Hamming modes. "
                          "This is unexpected; continuing but splits may be inconsistent.")

    segments_by_hamming[hmode] = (segments, subject_to_seg_ids)

print("\n[INFO] Done precomputing.")

def segs_from_subjects(subject_to_seg_ids, subj_ids):
    out = []
    for sid in subj_ids:
        out.extend(subject_to_seg_ids.get(sid, []))
    return sorted(out)

# =========================
# Switch sweep
# =========================
SWITCH_SIM_SIGNAL = ["raw", "filtered"]               # S1
SWITCH_THRESH_TUNE_LEVEL = ["segment", "subject"]    # S3

def run_one_config(hmode: str, sim_signal: str, thresh_tune_level: str):
    print("\n" + "="*90)
    print(f"[CONFIG] HAMMING_MODE={hmode} | SIM_SIGNAL={sim_signal} | THRESH_TUNE_LEVEL={thresh_tune_level}")
    print("="*90)

    set_seed(SEED)

    segments, subject_to_seg_ids = segments_by_hamming[hmode]

    # build seg id lists for this hmode
    mdd_tr_seg = segs_from_subjects(subject_to_seg_ids, mdd_tr)
    mdd_va_seg = segs_from_subjects(subject_to_seg_ids, mdd_va)
    mdd_te_seg = segs_from_subjects(subject_to_seg_ids, mdd_te)
    hc_tr_seg  = segs_from_subjects(subject_to_seg_ids, hc_tr)
    hc_va_seg  = segs_from_subjects(subject_to_seg_ids, hc_va)
    hc_te_seg  = segs_from_subjects(subject_to_seg_ids, hc_te)

    activate_sim_mode(segments, sim_signal=sim_signal)

    # Datasets/loaders (class-conditional)
    mdd_train_ds = GeneratorDataset(segments, mdd_tr_seg)
    hc_train_ds  = GeneratorDataset(segments, hc_tr_seg)

    mdd_loader = DataLoader(mdd_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
    hc_loader  = DataLoader(hc_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)

    gen_mdd = GDNGenerator(in_channels=K_SIM).to(DEVICE)
    gen_hc  = GDNGenerator(in_channels=K_SIM).to(DEVICE)

    print("[TRAIN] GMDD")
    gen_mdd = train_generator(gen_mdd, mdd_loader, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY)
    print("[TRAIN] GHC")
    gen_hc  = train_generator(gen_hc,  hc_loader,  epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY)

    # Validation: threshold selection
    val_ids = mdd_va_seg + hc_va_seg
    print("[EVAL] validation segments:", len(val_ids))
    val_res = evaluate_segments(gen_mdd, gen_hc, segments, val_ids)

    if thresh_tune_level == "segment":
        n0, score = find_best_threshold_segment(val_res)
        tune_score_name = "val_seg_acc"
    elif thresh_tune_level == "subject":
        n0, score = find_best_threshold_subject(val_res)
        tune_score_name = "val_subj_acc"
    else:
        raise ValueError("Unknown thresh_tune_level: " + str(thresh_tune_level))

    print(f"[THRESH] selected n0={n0} ({tune_score_name}={score*100:.2f}%)")

    # Test
    test_ids = mdd_te_seg + hc_te_seg
    print("[EVAL] test segments:", len(test_ids))
    test_res = evaluate_segments(gen_mdd, gen_hc, segments, test_ids)

    m_seg = metrics_segment(test_res, n0)
    subj_acc, subj_cm = subject_level_from_results(test_res, n0)

    row = {
        "HAMMING_MODE": hmode,
        "SIM_SIGNAL": sim_signal,
        "THRESH_TUNE_LEVEL": thresh_tune_level,
        "n0": n0,
        "val_tune_score": score,
        "seg_acc": m_seg["acc"],
        "seg_bal_acc": m_seg["bal_acc"],
        "seg_sens": m_seg["sens"],
        "seg_spec": m_seg["spec"],
        "subj_acc": subj_acc,
        "subj_tp": subj_cm["tp"],
        "subj_tn": subj_cm["tn"],
        "subj_fp": subj_cm["fp"],
        "subj_fn": subj_cm["fn"],
    }

    del gen_mdd, gen_hc, mdd_train_ds, hc_train_ds, mdd_loader, hc_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

rows = []
for hmode in HAMMING_LEVELS:
    for sim_signal in SWITCH_SIM_SIGNAL:
        for tune_level in SWITCH_THRESH_TUNE_LEVEL:
            rows.append(run_one_config(hmode, sim_signal, tune_level))

df = pd.DataFrame(rows)
df = df.sort_values(["HAMMING_MODE","SIM_SIGNAL","THRESH_TUNE_LEVEL"]).reset_index(drop=True)

print("\n\n====================")
print("Sweep results (subject-level accuracy is the headline):")
print("====================")
display(df)

subj_min = float(df["subj_acc"].min())
subj_max = float(df["subj_acc"].max())
print(f"\n[SUMMARY] subject-acc range: {subj_min*100:.2f}% 〜 {subj_max*100:.2f}%  (Δ={ (subj_max-subj_min)*100:.2f} pts)")

seg_min = float(df["seg_acc"].min())
seg_max = float(df["seg_acc"].max())
print(f"[SUMMARY] segment-acc range : {seg_min*100:.2f}% 〜 {seg_max*100:.2f}%  (Δ={ (seg_max-seg_min)*100:.2f} pts)")

out_csv = "/content/gdn_switch_sweep_results.csv"
df.to_csv(out_csv, index=False)
print("[INFO] saved:", out_csv)

[pip] installing pywavelets ...
[pip] installing scikit-learn ...
[INFO] device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Unzipping zip -> /content/modma_lanzhou
[INFO] EXPECTED_WLEN = 1255 (db6, symmetric, seg_len=2500 => typically 1255)
[INFO] Unique subject IDs: 53
[INFO] label heuristic counts: MDD=24, HC=29
[INFO] subjects train/val/test: 15 5 4 | 15 5 9

[INFO] Precomputing per-segment features for each Hamming mode ...

--- Hamming mode = paper_plus_2pi ---


  0%|          | 0/53 [00:00<?, ?it/s]


--- Hamming mode = numpy_hamming ---


  0%|          | 0/53 [00:00<?, ?it/s]


[INFO] Done precomputing.

[CONFIG] HAMMING_MODE=paper_plus_2pi | SIM_SIGNAL=raw | THRESH_TUNE_LEVEL=segment
[TRAIN] GMDD
  [train] ep 01/6 loss=344.6202
  [train] ep 02/6 loss=245.9269
  [train] ep 04/6 loss=249.5877
  [train] ep 06/6 loss=233.0012
[TRAIN] GHC
  [train] ep 01/6 loss=446.3947
  [train] ep 02/6 loss=246.2185
  [train] ep 04/6 loss=237.4115
  [train] ep 06/6 loss=232.5981
[EVAL] validation segments: 80
[THRESH] selected n0=82 (val_seg_acc=61.25%)
[EVAL] test segments: 104

[CONFIG] HAMMING_MODE=paper_plus_2pi | SIM_SIGNAL=raw | THRESH_TUNE_LEVEL=subject
[TRAIN] GMDD
  [train] ep 01/6 loss=344.6202
  [train] ep 02/6 loss=245.9269
  [train] ep 04/6 loss=249.5877
  [train] ep 06/6 loss=233.0012
[TRAIN] GHC
  [train] ep 01/6 loss=446.3947
  [train] ep 02/6 loss=246.2185
  [train] ep 04/6 loss=237.4115
  [train] ep 06/6 loss=232.5981
[EVAL] validation segments: 80
[THRESH] selected n0=73 (val_subj_acc=70.00%)
[EVAL] test segments: 104

[CONFIG] HAMMING_MODE=paper_plus_2pi | 

,HAMMING_MODE,SIM_SIGNAL,THRESH_TUNE_LEVEL,n0,val_tune_score,seg_acc,seg_bal_acc,seg_sens,seg_spec,subj_acc,subj_tp,subj_tn,subj_fp,subj_fn
0,numpy_hamming,filtered,segment,62,0.6000,0.653846,0.515625,0.15625,0.875000,0.615385,0,8,1,4
1,numpy_hamming,filtered,subject,46,0.7000,0.596154,0.560764,0.46875,0.652778,0.615385,2,6,3,2
2,numpy_hamming,raw,segment,89,0.5750,0.480769,0.598958,0.90625,0.291667,0.461538,4,2,7,0
3,numpy_hamming,raw,subject,89,0.6000,0.480769,0.598958,0.90625,0.291667,0.461538,4,2,7,0
4,paper_plus_2pi,filtered,segment,52,0.6125,0.490385,0.406250,0.18750,0.625000,0.461538,0,6,3,4
5,paper_plus_2pi,filtered,subject,16,0.6000,0.326923,0.505208,0.96875,0.041667,0.307692,4,0,9,0
6,paper_plus_2pi,raw,segment,82,0.6125,0.663462,0.574653,0.34375,0.805556,0.692308,2,7,2,2
7,paper_plus_2pi,raw,subject,73,0.7000,0.557692,0.576389,0.62500,0.527778,0.538462,3,4,5,1



[SUMMARY] subject-acc range: 30.77% 〜 69.23%  (Δ=38.46 pts)
[SUMMARY] segment-acc range : 32.69% 〜 66.35%  (Δ=33.65 pts)
[INFO] saved: /content/gdn_switch_sweep_results.csv
